# **Import Important Libraries**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl

In [ ]:
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdmolops
from rdkit import Chem
from mordred import Calculator, descriptors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

# **Data Handling and Preprocessing**

In [ ]:
useless_cols = [   
    
    'MaxPartialCharge', 
    # Nan data
    'BCUT2D_MWHI',
    'BCUT2D_MWLOW',
    'BCUT2D_CHGHI',
    'BCUT2D_CHGLO',
    'BCUT2D_LOGPHI',
    'BCUT2D_LOGPLOW',
    'BCUT2D_MRHI',
    'BCUT2D_MRLOW',

    # Constant data
    'NumRadicalElectrons',
    'SMR_VSA8',
    'SlogP_VSA9',
    'fr_barbitur',
    'fr_benzodiazepine',
    'fr_dihydropyridine',
    'fr_epoxide',
    'fr_isothiocyan',
    'fr_lactam',
    'fr_nitroso',
    'fr_prisulfonamd',
    'fr_thiocyan',

    # High correlated data >0.95
    'MaxEStateIndex',
    'HeavyAtomMolWt',
    'ExactMolWt',
    'NumValenceElectrons',
    'Chi0',
    'Chi0n',
    'Chi0v',
    'Chi1',
    'Chi1n',
    'Chi1v',
    'Chi2n',
    'Kappa1',
    'LabuteASA',
    'HeavyAtomCount',
    'MolMR',
    'Chi3n',
    'BertzCT',
    'Chi2v',
    'Chi4n',
    'HallKierAlpha',
    'Chi3v',
    'Chi4v',
    'MinAbsPartialCharge',
    'MinPartialCharge',
    'MaxAbsPartialCharge',
    'FpDensityMorgan2',
    'FpDensityMorgan3',
    'Phi',
    'Kappa3',
    'fr_nitrile',
    'SlogP_VSA6',
    'NumAromaticCarbocycles',
    'NumAromaticRings',
    'fr_benzene',
    'VSA_EState6',
    'NOCount',
    'fr_C_O',
    'fr_C_O_noCOO',
    'NumHDonors',
    'fr_amide',
    'fr_Nhpyrrole',
    'fr_phenol',
    'fr_phenol_noOrthoHbond',
    'fr_COO2',
    'fr_halogen',
    'fr_diazo',
    'fr_nitro_arom',
    'fr_phos_ester'
]

In [ ]:
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

In [ ]:
tg = pd.read_csv('/kaggle/input/modred-dataset/desc_tg.csv',low_memory=False)
tc = pd.read_csv('/kaggle/input/modred-dataset/desc_tc.csv',low_memory=False)
rg = pd.read_csv('/kaggle/input/modred-dataset/desc_rg.csv',low_memory=False)
ffv = pd.read_csv('/kaggle/input/modred-dataset/desc_ffv.csv',low_memory=False)
density = pd.read_csv('/kaggle/input/modred-dataset/desc_de.csv',low_memory=False)
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv',low_memory=False)
Id= test['id']

In [ ]:
for i in (tg,tc,rg,ffv,density):
     i.drop(columns=[col for col in i.columns if i[col].nunique() == 1],axis=1,inplace=True)

In [ ]:
tg = tg.select_dtypes(exclude=['object', 'category'])
rg = rg.select_dtypes(exclude=['object', 'category'])
ffv = ffv.select_dtypes(exclude=['object', 'category'])
tc = tc.select_dtypes(exclude=['object', 'category'])
density  = density.select_dtypes(exclude=['object', 'category'])

In [ ]:
test_mol = [Chem.MolFromSmiles(s) for s in test.SMILES]

In [ ]:
calc = Calculator(descriptors, ignore_3D=True)
desc_test = calc.pandas(test_mol)

In [ ]:
def make_smile_canonical(smile):
    try:
        mol = Chem.MolFromSmiles(smile)
        canon_smile = Chem.MolToSmiles(mol, canonical=True)
        return canon_smile
    except:
        return np.nan
test['SMILES'] = test['SMILES'].apply(lambda s: make_smile_canonical(s))

In [ ]:
def preprocessing(df):
    desc_names = [desc[0] for desc in Descriptors.descList if desc[0] not in useless_cols]
    descriptors = [compute_all_descriptors(smi) for smi in df['SMILES'].to_list()]

    graph_feats = {'graph_diameter': [], 'avg_shortest_path': [], 'num_cycles': []}
    for smile in df['SMILES']:
         compute_graph_features(smile, graph_feats)
        
    result = pd.concat(
        [
            pd.DataFrame(descriptors, columns=desc_names),
            pd.DataFrame(graph_feats)
        ],
        axis=1
    )

    result = result.replace([-np.inf, np.inf], np.nan)
    return result

In [ ]:
def compute_all_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [None] * len(desc_names)
    return [desc[1](mol) for desc in Descriptors.descList if desc[0] not in useless_cols]

def compute_graph_features(smiles, graph_feats):
    mol = Chem.MolFromSmiles(smiles)
    adj = rdmolops.GetAdjacencyMatrix(mol)
    G = nx.from_numpy_array(adj)

    graph_feats['graph_diameter'].append(nx.diameter(G) if nx.is_connected(G) else 0)
    graph_feats['avg_shortest_path'].append(nx.average_shortest_path_length(G) if nx.is_connected(G) else 0)
    graph_feats['num_cycles'].append(len(list(nx.cycle_basis(G))))

test = pd.concat([test, preprocessing(test)], axis=1)
test['Ipc']=np.log10(test['Ipc'])

test=test.drop(['id','SMILES'],axis=1)

# **using optina for getting optimal parameters and using top parameters for model ensembling**

In [ ]:
!pip install optuna

In [ ]:
import optuna
from optuna.trial import TrialState
from optuna.trial import FrozenTrial
import datetime
from optuna.distributions import IntDistribution, FloatDistribution

In [ ]:
def objective(trial, x, y):
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 2)
    
    n_estimators = trial.suggest_int('n_estimators', 200, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
    gamma = trial.suggest_int('gamma', 0, 5)
    reg_lambda = trial.suggest_int('reg_lambda', 1, 10)
    reg_alpha = trial.suggest_int('reg_alpha', 0, 10)
    
    Model = XGBRegressor(
        n_estimators = n_estimators,
        random_state=42, 
        learning_rate = learning_rate,
        max_depth = max_depth,
        gamma = gamma,
        reg_lambda = reg_lambda,
        reg_alpha = reg_alpha,
        min_child_weight = min_child_weight
    )

    Model.fit(X_train, y_train)
    y_pred = Model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    
    return mae

In [ ]:
def objective_light(trial, x, y):
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)
    
    params = {
        "objective" : 'mae',
        "metric" : 'mae',
        "n_estimators": trial.suggest_int('n_estimators', 200, 500),
        "learning_rate": trial.suggest_float('learning_rate', 0.01, 0.1),
        "num_leaves":trial.suggest_int('num_leaves', 20, 130),
        "feature_fraction": trial.suggest_float('feature_fraction', 0.4, 1),
        "bagging_fraction": trial.suggest_float('bagging_fraction', 0.4, 1),
        "bagging_freq": 1,
        "min_child_samples": trial.suggest_int('min_child_samples', 5, 50),
        "random_state": 42,
        "n_jobs":-1,
        "verbosity" : -1
    }
    Model = lgb.LGBMRegressor(**params)

    Model.fit(X_train, y_train)
    y_pred = Model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    
    return mae

# **Model Initilization**

In [ ]:
class Model_XG():
    def __init__(self, params):
        self.params = params
        self.model_best = XGBRegressor(**self.params)
        
    def train(self, X, y):
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
        self.model_best.fit(self.X_train, self.y_train)
        
    def evalution(self):
        self.pred = self.model_best.predict(self.X_test)
        return (mean_absolute_error(self.pred, self.y_test))
    
    def prediction(self, X):
        p = self.model_best.predict(X)
        return p

In [ ]:
class Model_light():
    def __init__(self, params):
        self.params = params
        self.model_best = lgb.LGBMRegressor(**self.params)
        
    def train(self, X, y):
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
        self.model_best.fit(self.X_train, self.y_train)
        
    def evalution(self):
        self.pred = self.model_best.predict(self.X_test)
        return (mean_absolute_error(self.pred, self.y_test))
    
    def prediction(self, X):
        p = self.model_best.predict(X)
        return p

In [ ]:
object_cols = desc_test.select_dtypes(include=['object']).columns
desc_test[object_cols] = desc_test[object_cols].apply(pd.to_numeric, errors='coerce')
desc_test.fillna(0, inplace=True)

# **Model Training**
**for all the variables (Tg, Density, Tc, FFV, Rg)**

**TG**

In [ ]:
train_cols1 = set(tg.columns) - {"Tg"}
test_cols1 = set(desc_test.columns)
common_cols1 = list(train_cols1 & test_cols1)
x_tg=tg[common_cols1].copy()
y_tg=tg["Tg"].copy()

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective(trial, x_tg, y_tg), n_trials = 50)

In [ ]:
# completed_trials_tg = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_tg = sorted(completed_trials_tg, key =lambda t: t.value)

In [ ]:
sorted_tg = [FrozenTrial(number=43, state=1, values=[22.794370645992068], datetime_start=datetime.datetime(2025, 8, 5, 12, 2, 48, 6261), datetime_complete=datetime.datetime(2025, 8, 5, 12, 3, 56, 588907), params={'n_estimators': 464, 'learning_rate': 0.07796963163387444, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=43, value=None),
 FrozenTrial(number=49, state=1, values=[22.821911478512266], datetime_start=datetime.datetime(2025, 8, 5, 12, 9, 29, 457243), datetime_complete=datetime.datetime(2025, 8, 5, 12, 10, 47, 477211), params={'n_estimators': 489, 'learning_rate': 0.07452540801959374, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=49, value=None),
 FrozenTrial(number=40, state=1, values=[22.85815976176416], datetime_start=datetime.datetime(2025, 8, 5, 11, 58, 12, 954832), datetime_complete=datetime.datetime(2025, 8, 5, 11, 59, 48, 121665), params={'n_estimators': 468, 'learning_rate': 0.07665666164853398, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=46, state=1, values=[22.861009217188847], datetime_start=datetime.datetime(2025, 8, 5, 12, 5, 55, 142988), datetime_complete=datetime.datetime(2025, 8, 5, 12, 7, 14, 222908), params={'n_estimators': 485, 'learning_rate': 0.07366743698080623, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=46, value=None),
 FrozenTrial(number=41, state=1, values=[22.89592920551439], datetime_start=datetime.datetime(2025, 8, 5, 11, 59, 48, 122762), datetime_complete=datetime.datetime(2025, 8, 5, 12, 1, 15, 942724), params={'n_estimators': 470, 'learning_rate': 0.07816420609677943, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=41, value=None),
 FrozenTrial(number=48, state=1, values=[22.972021063575546], datetime_start=datetime.datetime(2025, 8, 5, 12, 8, 7, 188560), datetime_complete=datetime.datetime(2025, 8, 5, 12, 9, 29, 456177), params={'n_estimators': 500, 'learning_rate': 0.0718856075553141, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=48, value=None),
 FrozenTrial(number=28, state=1, values=[22.984394141625668], datetime_start=datetime.datetime(2025, 8, 5, 11, 44, 47, 774137), datetime_complete=datetime.datetime(2025, 8, 5, 11, 45, 58, 117308), params={'n_estimators': 434, 'learning_rate': 0.06819685383802536, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 7, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=28, value=None),
 FrozenTrial(number=44, state=1, values=[22.992593196365018], datetime_start=datetime.datetime(2025, 8, 5, 12, 3, 56, 590077), datetime_complete=datetime.datetime(2025, 8, 5, 12, 5, 7, 529104), params={'n_estimators': 466, 'learning_rate': 0.07773535964083239, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=44, value=None),
 FrozenTrial(number=47, state=1, values=[23.061537298869137], datetime_start=datetime.datetime(2025, 8, 5, 12, 7, 14, 223961), datetime_complete=datetime.datetime(2025, 8, 5, 12, 8, 7, 187447), params={'n_estimators': 487, 'learning_rate': 0.08703681050737448, 'max_depth': 7, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=47, value=None),
 FrozenTrial(number=42, state=1, values=[23.073056783082876], datetime_start=datetime.datetime(2025, 8, 5, 12, 1, 15, 943870), datetime_complete=datetime.datetime(2025, 8, 5, 12, 2, 48, 5125), params={'n_estimators': 470, 'learning_rate': 0.07739646548275209, 'max_depth': 9, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=42, value=None),
 FrozenTrial(number=23, state=1, values=[23.086863823728883], datetime_start=datetime.datetime(2025, 8, 5, 11, 40, 3, 154954), datetime_complete=datetime.datetime(2025, 8, 5, 11, 41, 0, 521482), params={'n_estimators': 440, 'learning_rate': 0.09221079416663133, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=23, value=None),
 FrozenTrial(number=31, state=1, values=[23.16350975608724], datetime_start=datetime.datetime(2025, 8, 5, 11, 48, 19, 880441), datetime_complete=datetime.datetime(2025, 8, 5, 11, 49, 31, 147149), params={'n_estimators': 477, 'learning_rate': 0.0681487944444968, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 3, 'reg_lambda': 7, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=31, value=None),
 FrozenTrial(number=6, state=1, values=[23.193259730602318], datetime_start=datetime.datetime(2025, 8, 5, 11, 25, 40, 529293), datetime_complete=datetime.datetime(2025, 8, 5, 11, 26, 47, 155014), params={'n_estimators': 340, 'learning_rate': 0.06165731796809145, 'max_depth': 10, 'min_child_weight': 8, 'gamma': 4, 'reg_lambda': 2, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=6, value=None),
 FrozenTrial(number=21, state=1, values=[23.198043187857596], datetime_start=datetime.datetime(2025, 8, 5, 11, 37, 32, 749576), datetime_complete=datetime.datetime(2025, 8, 5, 11, 38, 41, 112894), params={'n_estimators': 457, 'learning_rate': 0.08165740727930489, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=21, value=None),
 FrozenTrial(number=32, state=1, values=[23.21126800352968], datetime_start=datetime.datetime(2025, 8, 5, 11, 49, 31, 152783), datetime_complete=datetime.datetime(2025, 8, 5, 11, 50, 48, 986927), params={'n_estimators': 480, 'learning_rate': 0.06707180544315958, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 8, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=32, value=None),
 FrozenTrial(number=17, state=1, values=[23.218042592434365], datetime_start=datetime.datetime(2025, 8, 5, 11, 35, 46, 224824), datetime_complete=datetime.datetime(2025, 8, 5, 11, 36, 31, 115789), params={'n_estimators': 451, 'learning_rate': 0.08434427032986055, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 3, 'reg_lambda': 6, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=17, value=None),
 FrozenTrial(number=26, state=1, values=[23.22231421685881], datetime_start=datetime.datetime(2025, 8, 5, 11, 42, 31, 345391), datetime_complete=datetime.datetime(2025, 8, 5, 11, 43, 18, 925332), params={'n_estimators': 389, 'learning_rate': 0.09208851899930587, 'max_depth': 8, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=26, value=None),
 FrozenTrial(number=37, state=1, values=[23.232079503617307], datetime_start=datetime.datetime(2025, 8, 5, 11, 55, 18, 43701), datetime_complete=datetime.datetime(2025, 8, 5, 11, 56, 50, 392142), params={'n_estimators': 499, 'learning_rate': 0.07252690136718251, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 3, 'reg_lambda': 5, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=37, value=None),
 FrozenTrial(number=24, state=1, values=[23.2572970801017], datetime_start=datetime.datetime(2025, 8, 5, 11, 41, 0, 522645), datetime_complete=datetime.datetime(2025, 8, 5, 11, 41, 54, 158457), params={'n_estimators': 422, 'learning_rate': 0.09306477760015985, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 5, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=38, state=1, values=[23.304057045506255], datetime_start=datetime.datetime(2025, 8, 5, 11, 56, 50, 393241), datetime_complete=datetime.datetime(2025, 8, 5, 11, 57, 43, 376081), params={'n_estimators': 476, 'learning_rate': 0.06849735372268337, 'max_depth': 8, 'min_child_weight': 9, 'gamma': 5, 'reg_lambda': 5, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=38, value=None),
 FrozenTrial(number=22, state=1, values=[23.325578043607127], datetime_start=datetime.datetime(2025, 8, 5, 11, 38, 41, 113999), datetime_complete=datetime.datetime(2025, 8, 5, 11, 40, 3, 153809), params={'n_estimators': 470, 'learning_rate': 0.0813584909739668, 'max_depth': 10, 'min_child_weight': 6, 'gamma': 3, 'reg_lambda': 8, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=22, value=None),
 FrozenTrial(number=33, state=1, values=[23.329081502955656], datetime_start=datetime.datetime(2025, 8, 5, 11, 50, 48, 988082), datetime_complete=datetime.datetime(2025, 8, 5, 11, 52, 16, 190152), params={'n_estimators': 495, 'learning_rate': 0.0691879748399224, 'max_depth': 10, 'min_child_weight': 7, 'gamma': 3, 'reg_lambda': 9, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=33, value=None),
 FrozenTrial(number=27, state=1, values=[23.354467704777363], datetime_start=datetime.datetime(2025, 8, 5, 11, 43, 18, 926343), datetime_complete=datetime.datetime(2025, 8, 5, 11, 44, 47, 773053), params={'n_estimators': 355, 'learning_rate': 0.05560515522022436, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=27, value=None),
 FrozenTrial(number=9, state=1, values=[23.390527898766084], datetime_start=datetime.datetime(2025, 8, 5, 11, 28, 14, 67015), datetime_complete=datetime.datetime(2025, 8, 5, 11, 29, 17, 979196), params={'n_estimators': 460, 'learning_rate': 0.07369893946243078, 'max_depth': 9, 'min_child_weight': 8, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=9, value=None),
 FrozenTrial(number=11, state=1, values=[23.458900878776603], datetime_start=datetime.datetime(2025, 8, 5, 11, 31, 29, 869594), datetime_complete=datetime.datetime(2025, 8, 5, 11, 32, 31, 317466), params={'n_estimators': 428, 'learning_rate': 0.077851692590019, 'max_depth': 10, 'min_child_weight': 8, 'gamma': 4, 'reg_lambda': 2, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=11, value=None),
 FrozenTrial(number=45, state=1, values=[23.50580339093295], datetime_start=datetime.datetime(2025, 8, 5, 12, 5, 7, 530206), datetime_complete=datetime.datetime(2025, 8, 5, 12, 5, 55, 141810), params={'n_estimators': 462, 'learning_rate': 0.0784142732638363, 'max_depth': 7, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=45, value=None),
 FrozenTrial(number=35, state=1, values=[23.554299024189245], datetime_start=datetime.datetime(2025, 8, 5, 11, 52, 54, 701206), datetime_complete=datetime.datetime(2025, 8, 5, 11, 54, 35, 83757), params={'n_estimators': 431, 'learning_rate': 0.05977148710918149, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 4, 'reg_lambda': 7, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=35, value=None),
 FrozenTrial(number=5, state=1, values=[23.64716730350103], datetime_start=datetime.datetime(2025, 8, 5, 11, 24, 45, 870338), datetime_complete=datetime.datetime(2025, 8, 5, 11, 25, 40, 528211), params={'n_estimators': 488, 'learning_rate': 0.046732196134831386, 'max_depth': 8, 'min_child_weight': 10, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=5, value=None),
 FrozenTrial(number=15, state=1, values=[23.65668504081035], datetime_start=datetime.datetime(2025, 8, 5, 11, 34, 44, 440204), datetime_complete=datetime.datetime(2025, 8, 5, 11, 35, 27, 429173), params={'n_estimators': 262, 'learning_rate': 0.06229342873831668, 'max_depth': 9, 'min_child_weight': 7, 'gamma': 4, 'reg_lambda': 1, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=15, value=None),
 FrozenTrial(number=7, state=1, values=[23.7030036539941], datetime_start=datetime.datetime(2025, 8, 5, 11, 26, 47, 156150), datetime_complete=datetime.datetime(2025, 8, 5, 11, 27, 24, 816241), params={'n_estimators': 377, 'learning_rate': 0.08628757557826715, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=7, value=None),
 FrozenTrial(number=29, state=1, values=[23.880700036997744], datetime_start=datetime.datetime(2025, 8, 5, 11, 45, 58, 118393), datetime_complete=datetime.datetime(2025, 8, 5, 11, 47, 4, 168328), params={'n_estimators': 374, 'learning_rate': 0.04696323729511112, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 9, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=29, value=None),
 FrozenTrial(number=34, state=1, values=[24.059187655208508], datetime_start=datetime.datetime(2025, 8, 5, 11, 52, 16, 191231), datetime_complete=datetime.datetime(2025, 8, 5, 11, 52, 54, 700087), params={'n_estimators': 400, 'learning_rate': 0.06209636268949012, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 3, 'reg_lambda': 7, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=34, value=None),
 FrozenTrial(number=14, state=1, values=[24.16108685608866], datetime_start=datetime.datetime(2025, 8, 5, 11, 33, 51, 391381), datetime_complete=datetime.datetime(2025, 8, 5, 11, 34, 44, 439092), params={'n_estimators': 331, 'learning_rate': 0.0353206003699496, 'max_depth': 9, 'min_child_weight': 9, 'gamma': 4, 'reg_lambda': 3, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=14, value=None),
 FrozenTrial(number=12, state=1, values=[24.307559476780874], datetime_start=datetime.datetime(2025, 8, 5, 11, 32, 31, 318664), datetime_complete=datetime.datetime(2025, 8, 5, 11, 33, 37, 401273), params={'n_estimators': 312, 'learning_rate': 0.02776949914394545, 'max_depth': 10, 'min_child_weight': 8, 'gamma': 3, 'reg_lambda': 3, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=12, value=None),
 FrozenTrial(number=8, state=1, values=[24.314281168438914], datetime_start=datetime.datetime(2025, 8, 5, 11, 27, 24, 817307), datetime_complete=datetime.datetime(2025, 8, 5, 11, 28, 14, 65913), params={'n_estimators': 407, 'learning_rate': 0.04964148723951063, 'max_depth': 8, 'min_child_weight': 8, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=0, state=1, values=[24.3263387936532], datetime_start=datetime.datetime(2025, 8, 5, 11, 22, 20, 219843), datetime_complete=datetime.datetime(2025, 8, 5, 11, 23, 6, 87313), params={'n_estimators': 249, 'learning_rate': 0.04710132708683262, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 7, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=36, state=1, values=[24.370302304400926], datetime_start=datetime.datetime(2025, 8, 5, 11, 54, 35, 84857), datetime_complete=datetime.datetime(2025, 8, 5, 11, 55, 18, 42627), params={'n_estimators': 338, 'learning_rate': 0.05052543962182935, 'max_depth': 8, 'min_child_weight': 6, 'gamma': 4, 'reg_lambda': 10, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=36, value=None),
 FrozenTrial(number=3, state=1, values=[24.448653477713446], datetime_start=datetime.datetime(2025, 8, 5, 11, 23, 49, 928650), datetime_complete=datetime.datetime(2025, 8, 5, 11, 24, 23, 577931), params={'n_estimators': 491, 'learning_rate': 0.0657636598001003, 'max_depth': 6, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 5, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=3, value=None),
 FrozenTrial(number=18, state=1, values=[24.58819609768413], datetime_start=datetime.datetime(2025, 8, 5, 11, 36, 31, 116892), datetime_complete=datetime.datetime(2025, 8, 5, 11, 37, 5, 501591), params={'n_estimators': 336, 'learning_rate': 0.056438914584357486, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 3, 'reg_lambda': 3, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=18, value=None),
 FrozenTrial(number=1, state=1, values=[24.660432850699916], datetime_start=datetime.datetime(2025, 8, 5, 11, 23, 6, 88428), datetime_complete=datetime.datetime(2025, 8, 5, 11, 23, 41, 23670), params={'n_estimators': 229, 'learning_rate': 0.0710235896096664, 'max_depth': 8, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 10, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=1, value=None),
 FrozenTrial(number=10, state=1, values=[24.69767408745193], datetime_start=datetime.datetime(2025, 8, 5, 11, 29, 17, 980322), datetime_complete=datetime.datetime(2025, 8, 5, 11, 31, 29, 868522), params={'n_estimators': 302, 'learning_rate': 0.019018962667219773, 'max_depth': 10, 'min_child_weight': 1, 'gamma': 5, 'reg_lambda': 1, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=10, value=None),
 FrozenTrial(number=4, state=1, values=[25.32250094125074], datetime_start=datetime.datetime(2025, 8, 5, 11, 24, 23, 579644), datetime_complete=datetime.datetime(2025, 8, 5, 11, 24, 45, 869249), params={'n_estimators': 362, 'learning_rate': 0.06423410427422722, 'max_depth': 6, 'min_child_weight': 10, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=4, value=None),
 FrozenTrial(number=25, state=1, values=[25.38788190873859], datetime_start=datetime.datetime(2025, 8, 5, 11, 41, 54, 159672), datetime_complete=datetime.datetime(2025, 8, 5, 11, 42, 31, 344231), params={'n_estimators': 200, 'learning_rate': 0.03858520431498198, 'max_depth': 9, 'min_child_weight': 7, 'gamma': 2, 'reg_lambda': 8, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=25, value=None),
 FrozenTrial(number=19, state=1, values=[25.745198131788758], datetime_start=datetime.datetime(2025, 8, 5, 11, 37, 5, 502681), datetime_complete=datetime.datetime(2025, 8, 5, 11, 37, 26, 33006), params={'n_estimators': 411, 'learning_rate': 0.08534309764535006, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 4, 'reg_lambda': 7, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=19, value=None),
 FrozenTrial(number=39, state=1, values=[25.857135175156728], datetime_start=datetime.datetime(2025, 8, 5, 11, 57, 43, 377208), datetime_complete=datetime.datetime(2025, 8, 5, 11, 58, 12, 953702), params={'n_estimators': 445, 'learning_rate': 0.038761418894722746, 'max_depth': 6, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=39, value=None),
 FrozenTrial(number=16, state=1, values=[25.929360514391462], datetime_start=datetime.datetime(2025, 8, 5, 11, 35, 27, 430237), datetime_complete=datetime.datetime(2025, 8, 5, 11, 35, 46, 223771), params={'n_estimators': 388, 'learning_rate': 0.07445060339507494, 'max_depth': 5, 'min_child_weight': 9, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=13, state=1, values=[27.076902992054748], datetime_start=datetime.datetime(2025, 8, 5, 11, 33, 37, 402362), datetime_complete=datetime.datetime(2025, 8, 5, 11, 33, 51, 390270), params={'n_estimators': 445, 'learning_rate': 0.0993197886659104, 'max_depth': 4, 'min_child_weight': 7, 'gamma': 5, 'reg_lambda': 6, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=13, value=None),
 FrozenTrial(number=30, state=1, values=[27.184666857710358], datetime_start=datetime.datetime(2025, 8, 5, 11, 47, 4, 173056), datetime_complete=datetime.datetime(2025, 8, 5, 11, 48, 19, 879338), params={'n_estimators': 431, 'learning_rate': 0.011137601666608786, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 5, 'reg_lambda': 7, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=30, value=None),
 FrozenTrial(number=2, state=1, values=[30.57801991821208], datetime_start=datetime.datetime(2025, 8, 5, 11, 23, 41, 24797), datetime_complete=datetime.datetime(2025, 8, 5, 11, 23, 49, 927028), params={'n_estimators': 305, 'learning_rate': 0.09013050951557733, 'max_depth': 3, 'min_child_weight': 2, 'gamma': 3, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=2, value=None),
 FrozenTrial(number=20, state=1, values=[30.640951982737157], datetime_start=datetime.datetime(2025, 8, 5, 11, 37, 26, 34115), datetime_complete=datetime.datetime(2025, 8, 5, 11, 37, 32, 748433), params={'n_estimators': 272, 'learning_rate': 0.09860392695615369, 'max_depth': 3, 'min_child_weight': 1, 'gamma': 5, 'reg_lambda': 4, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=20, value=None)]


In [ ]:
desc_tg = desc_test[common_cols1]
top_trials = sorted_tg[:10]
all_predictions = []
TG = []
for i, trial in enumerate(top_trials):
    params = trial.params
    params['random_state'] = 42+i
    model = Model_XG(params)
    model.train(x_tg, y_tg)
    preds = model.evalution()
    all_predictions.append(preds)
    TG.append(model.prediction(desc_tg))
    print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
final_ensemble_prediction = np.mean(all_predictions, axis=0)
final_TG = np.mean(TG, axis=0)
print(final_ensemble_prediction)
print(final_TG)

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective_light(trial, x_tg, y_tg), n_trials = 50)

In [ ]:
# completed_trials_tg_light = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_tg_light = sorted(completed_trials_tg_light, key =lambda t: t.value)

In [ ]:
# sorted_tg_light = [FrozenTrial(number=31, state=1, values=[26.158591668070386], datetime_start=datetime.datetime(2025, 8, 5, 12, 34, 8, 599830), datetime_complete=datetime.datetime(2025, 8, 5, 12, 34, 32, 992414), params={'n_estimators': 354, 'learning_rate': 0.09035929288976138, 'num_leaves': 114, 'feature_fraction': 0.6417842843932315, 'bagging_fraction': 0.7512073396338251, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=31, value=None),
#  FrozenTrial(number=32, state=1, values=[26.20551173275722], datetime_start=datetime.datetime(2025, 8, 5, 12, 34, 32, 995247), datetime_complete=datetime.datetime(2025, 8, 5, 12, 35, 1, 683373), params={'n_estimators': 353, 'learning_rate': 0.07648828895965284, 'num_leaves': 117, 'feature_fraction': 0.7981532142787963, 'bagging_fraction': 0.7531768875921229, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=32, value=None),
#  FrozenTrial(number=18, state=1, values=[26.253804325837322], datetime_start=datetime.datetime(2025, 8, 5, 12, 29, 3, 955412), datetime_complete=datetime.datetime(2025, 8, 5, 12, 29, 39, 76596), params={'n_estimators': 400, 'learning_rate': 0.08479271849459058, 'num_leaves': 107, 'feature_fraction': 0.9174311879148425, 'bagging_fraction': 0.7301303862421197, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=18, value=None),
#  FrozenTrial(number=34, state=1, values=[26.27716696189068], datetime_start=datetime.datetime(2025, 8, 5, 12, 35, 19, 403099), datetime_complete=datetime.datetime(2025, 8, 5, 12, 35, 48, 381217), params={'n_estimators': 344, 'learning_rate': 0.09124470885224661, 'num_leaves': 115, 'feature_fraction': 0.809381148830913, 'bagging_fraction': 0.5901596768049516, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=34, value=None),
#  FrozenTrial(number=45, state=1, values=[26.427441346290326], datetime_start=datetime.datetime(2025, 8, 5, 12, 39, 34, 997985), datetime_complete=datetime.datetime(2025, 8, 5, 12, 40, 12, 246936), params={'n_estimators': 382, 'learning_rate': 0.07534430946270097, 'num_leaves': 116, 'feature_fraction': 0.9391265073317505, 'bagging_fraction': 0.633564389119233, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=45, value=None),
#  FrozenTrial(number=28, state=1, values=[26.48053234467785], datetime_start=datetime.datetime(2025, 8, 5, 12, 32, 52, 383119), datetime_complete=datetime.datetime(2025, 8, 5, 12, 33, 23, 273602), params={'n_estimators': 346, 'learning_rate': 0.07756727699930444, 'num_leaves': 130, 'feature_fraction': 0.7532611782485173, 'bagging_fraction': 0.5568427098511347, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=28, value=None),
#  FrozenTrial(number=22, state=1, values=[26.505999642361672], datetime_start=datetime.datetime(2025, 8, 5, 12, 31, 4, 336711), datetime_complete=datetime.datetime(2025, 8, 5, 12, 31, 24, 753329), params={'n_estimators': 332, 'learning_rate': 0.08318405105788937, 'num_leaves': 130, 'feature_fraction': 0.5059814758013279, 'bagging_fraction': 0.716287032609729, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=22, value=None),
#  FrozenTrial(number=42, state=1, values=[26.511998850508704], datetime_start=datetime.datetime(2025, 8, 5, 12, 38, 29, 873946), datetime_complete=datetime.datetime(2025, 8, 5, 12, 38, 56, 939783), params={'n_estimators': 348, 'learning_rate': 0.07934226950181905, 'num_leaves': 119, 'feature_fraction': 0.7761317659020154, 'bagging_fraction': 0.5210380227331702, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=42, value=None),
#  FrozenTrial(number=20, state=1, values=[26.51516868617631], datetime_start=datetime.datetime(2025, 8, 5, 12, 30, 7, 846900), datetime_complete=datetime.datetime(2025, 8, 5, 12, 30, 33, 272025), params={'n_estimators': 398, 'learning_rate': 0.0878012001963307, 'num_leaves': 109, 'feature_fraction': 0.6641577475201281, 'bagging_fraction': 0.7051820119818615, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=20, value=None),
#  FrozenTrial(number=26, state=1, values=[26.537210687025848], datetime_start=datetime.datetime(2025, 8, 5, 12, 32, 9, 787203), datetime_complete=datetime.datetime(2025, 8, 5, 12, 32, 29, 784109), params={'n_estimators': 351, 'learning_rate': 0.0914707552852605, 'num_leaves': 121, 'feature_fraction': 0.5400434046764804, 'bagging_fraction': 0.6464337115296811, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=26, value=None),
#  FrozenTrial(number=21, state=1, values=[26.551470517459954], datetime_start=datetime.datetime(2025, 8, 5, 12, 30, 33, 274819), datetime_complete=datetime.datetime(2025, 8, 5, 12, 31, 4, 334953), params={'n_estimators': 414, 'learning_rate': 0.08364288327488391, 'num_leaves': 112, 'feature_fraction': 0.6725183335571382, 'bagging_fraction': 0.709235703031106, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=21, value=None),
#  FrozenTrial(number=36, state=1, values=[26.597343098912358], datetime_start=datetime.datetime(2025, 8, 5, 12, 36, 11, 908359), datetime_complete=datetime.datetime(2025, 8, 5, 12, 36, 39, 331629), params={'n_estimators': 374, 'learning_rate': 0.0994937779515501, 'num_leaves': 101, 'feature_fraction': 0.8178551989882492, 'bagging_fraction': 0.6769540423701231, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=36, value=None),
#  FrozenTrial(number=48, state=1, values=[26.605679510951088], datetime_start=datetime.datetime(2025, 8, 5, 12, 41, 5, 737104), datetime_complete=datetime.datetime(2025, 8, 5, 12, 41, 34, 495428), params={'n_estimators': 418, 'learning_rate': 0.08972356284231671, 'num_leaves': 88, 'feature_fraction': 0.886137006903976, 'bagging_fraction': 0.6808628441742153, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=48, value=None),
#  FrozenTrial(number=19, state=1, values=[26.610303432019656], datetime_start=datetime.datetime(2025, 8, 5, 12, 29, 39, 79153), datetime_complete=datetime.datetime(2025, 8, 5, 12, 30, 7, 843821), params={'n_estimators': 411, 'learning_rate': 0.08754908311069304, 'num_leaves': 110, 'feature_fraction': 0.6884550056810661, 'bagging_fraction': 0.6971938468647105, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=19, value=None),
#  FrozenTrial(number=27, state=1, values=[26.647425346324034], datetime_start=datetime.datetime(2025, 8, 5, 12, 32, 29, 785613), datetime_complete=datetime.datetime(2025, 8, 5, 12, 32, 52, 379761), params={'n_estimators': 399, 'learning_rate': 0.08555003959731952, 'num_leaves': 105, 'feature_fraction': 0.6429667662793862, 'bagging_fraction': 0.8363268882011484, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=27, value=None),
#  FrozenTrial(number=49, state=1, values=[26.666333310012885], datetime_start=datetime.datetime(2025, 8, 5, 12, 41, 34, 498393), datetime_complete=datetime.datetime(2025, 8, 5, 12, 42, 4, 433883), params={'n_estimators': 379, 'learning_rate': 0.083406831409648, 'num_leaves': 108, 'feature_fraction': 0.8467305500290901, 'bagging_fraction': 0.6049796685709645, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=49, value=None),
#  FrozenTrial(number=41, state=1, values=[26.69260924881354], datetime_start=datetime.datetime(2025, 8, 5, 12, 38, 2, 278252), datetime_complete=datetime.datetime(2025, 8, 5, 12, 38, 29, 871979), params={'n_estimators': 343, 'learning_rate': 0.07588075912046367, 'num_leaves': 124, 'feature_fraction': 0.7258906630118702, 'bagging_fraction': 0.5748457817843728, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=41, value=None),
#  FrozenTrial(number=14, state=1, values=[26.849672084889086], datetime_start=datetime.datetime(2025, 8, 5, 12, 27, 17, 810868), datetime_complete=datetime.datetime(2025, 8, 5, 12, 27, 49, 244467), params={'n_estimators': 473, 'learning_rate': 0.09878046197199, 'num_leaves': 93, 'feature_fraction': 0.9348810449339217, 'bagging_fraction': 0.8109263969452611, 'min_child_samples': 24}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=14, value=None),
#  FrozenTrial(number=23, state=1, values=[26.85936330743842], datetime_start=datetime.datetime(2025, 8, 5, 12, 31, 24, 756462), datetime_complete=datetime.datetime(2025, 8, 5, 12, 31, 42, 956373), params={'n_estimators': 337, 'learning_rate': 0.0643465874415309, 'num_leaves': 130, 'feature_fraction': 0.4864685064576819, 'bagging_fraction': 0.6623689215646482, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=23, value=None),
#  FrozenTrial(number=33, state=1, values=[26.859784064839626], datetime_start=datetime.datetime(2025, 8, 5, 12, 35, 1, 685065), datetime_complete=datetime.datetime(2025, 8, 5, 12, 35, 19, 400168), params={'n_estimators': 360, 'learning_rate': 0.07709452159313936, 'num_leaves': 74, 'feature_fraction': 0.8028940748025344, 'bagging_fraction': 0.7557164683891984, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=33, value=None),
#  FrozenTrial(number=25, state=1, values=[26.898860549397437], datetime_start=datetime.datetime(2025, 8, 5, 12, 31, 58, 404298), datetime_complete=datetime.datetime(2025, 8, 5, 12, 32, 9, 784689), params={'n_estimators': 256, 'learning_rate': 0.08019597494626965, 'num_leaves': 120, 'feature_fraction': 0.45426458070462283, 'bagging_fraction': 0.7362511600261282, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=25, value=None),
#  FrozenTrial(number=29, state=1, values=[26.90819082297973], datetime_start=datetime.datetime(2025, 8, 5, 12, 33, 23, 275219), datetime_complete=datetime.datetime(2025, 8, 5, 12, 33, 50, 652110), params={'n_estimators': 329, 'learning_rate': 0.06037322256580782, 'num_leaves': 130, 'feature_fraction': 0.7442770998079709, 'bagging_fraction': 0.5601420934316734, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=29, value=None),
#  FrozenTrial(number=24, state=1, values=[26.917202410865325], datetime_start=datetime.datetime(2025, 8, 5, 12, 31, 42, 959364), datetime_complete=datetime.datetime(2025, 8, 5, 12, 31, 58, 401785), params={'n_estimators': 305, 'learning_rate': 0.06184993842179656, 'num_leaves': 105, 'feature_fraction': 0.5271414158049741, 'bagging_fraction': 0.7285278788677341, 'min_child_samples': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=24, value=None),
#  FrozenTrial(number=46, state=1, values=[26.927192562079885], datetime_start=datetime.datetime(2025, 8, 5, 12, 40, 12, 248503), datetime_complete=datetime.datetime(2025, 8, 5, 12, 40, 42, 440657), params={'n_estimators': 384, 'learning_rate': 0.03837960009030014, 'num_leaves': 100, 'feature_fraction': 0.9497813981376684, 'bagging_fraction': 0.6379830725028883, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=46, value=None),
#  FrozenTrial(number=43, state=1, values=[26.970037449848267], datetime_start=datetime.datetime(2025, 8, 5, 12, 38, 56, 941209), datetime_complete=datetime.datetime(2025, 8, 5, 12, 39, 23, 395889), params={'n_estimators': 321, 'learning_rate': 0.06854738554627401, 'num_leaves': 124, 'feature_fraction': 0.8255035400129919, 'bagging_fraction': 0.5317776035064112, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=43, value=None),
#  FrozenTrial(number=12, state=1, values=[26.978876923049043], datetime_start=datetime.datetime(2025, 8, 5, 12, 26, 30, 648117), datetime_complete=datetime.datetime(2025, 8, 5, 12, 26, 56, 443736), params={'n_estimators': 497, 'learning_rate': 0.07287178287069492, 'num_leaves': 90, 'feature_fraction': 0.9913880612499796, 'bagging_fraction': 0.8013148633270202, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=12, value=None),
#  FrozenTrial(number=15, state=1, values=[26.97965257628199], datetime_start=datetime.datetime(2025, 8, 5, 12, 27, 49, 246063), datetime_complete=datetime.datetime(2025, 8, 5, 12, 28, 18, 623860), params={'n_estimators': 432, 'learning_rate': 0.09498744585369082, 'num_leaves': 99, 'feature_fraction': 0.8946332278619316, 'bagging_fraction': 0.8626469185879632, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=15, value=None),
#  FrozenTrial(number=16, state=1, values=[26.98420901093543], datetime_start=datetime.datetime(2025, 8, 5, 12, 28, 18, 625133), datetime_complete=datetime.datetime(2025, 8, 5, 12, 28, 48, 749477), params={'n_estimators': 464, 'learning_rate': 0.09809529533978632, 'num_leaves': 94, 'feature_fraction': 0.9161600606303775, 'bagging_fraction': 0.7008394442389454, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=16, value=None),
#  FrozenTrial(number=35, state=1, values=[26.991629137157968], datetime_start=datetime.datetime(2025, 8, 5, 12, 35, 48, 383610), datetime_complete=datetime.datetime(2025, 8, 5, 12, 36, 11, 906551), params={'n_estimators': 311, 'learning_rate': 0.08970616019432719, 'num_leaves': 116, 'feature_fraction': 0.8539279329241056, 'bagging_fraction': 0.6007979206626012, 'min_child_samples': 15}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=35, value=None),
#  FrozenTrial(number=11, state=1, values=[27.084829833584184], datetime_start=datetime.datetime(2025, 8, 5, 12, 26, 5, 913290), datetime_complete=datetime.datetime(2025, 8, 5, 12, 26, 30, 646015), params={'n_estimators': 500, 'learning_rate': 0.07210921530250576, 'num_leaves': 82, 'feature_fraction': 0.9659012822433626, 'bagging_fraction': 0.782269438439513, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=11, value=None),
#  FrozenTrial(number=10, state=1, values=[27.092024845180994], datetime_start=datetime.datetime(2025, 8, 5, 12, 25, 38, 487521), datetime_complete=datetime.datetime(2025, 8, 5, 12, 26, 5, 911753), params={'n_estimators': 500, 'learning_rate': 0.07124260084041986, 'num_leaves': 86, 'feature_fraction': 0.9808470807328054, 'bagging_fraction': 0.7927667930925996, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=10, value=None),
#  FrozenTrial(number=40, state=1, values=[27.11223665524125], datetime_start=datetime.datetime(2025, 8, 5, 12, 37, 42, 947890), datetime_complete=datetime.datetime(2025, 8, 5, 12, 38, 2, 275268), params={'n_estimators': 370, 'learning_rate': 0.04804525936075794, 'num_leaves': 79, 'feature_fraction': 0.7835013902731686, 'bagging_fraction': 0.6004608206766144, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=40, value=None),
#  FrozenTrial(number=39, state=1, values=[27.114539864720754], datetime_start=datetime.datetime(2025, 8, 5, 12, 37, 11, 836885), datetime_complete=datetime.datetime(2025, 8, 5, 12, 37, 42, 946500), params={'n_estimators': 383, 'learning_rate': 0.030837281949211683, 'num_leaves': 113, 'feature_fraction': 0.8695674728632137, 'bagging_fraction': 0.8307655629960565, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=39, value=None),
#  FrozenTrial(number=13, state=1, values=[27.129846287966142], datetime_start=datetime.datetime(2025, 8, 5, 12, 26, 56, 445522), datetime_complete=datetime.datetime(2025, 8, 5, 12, 27, 17, 808251), params={'n_estimators': 446, 'learning_rate': 0.08382720858448277, 'num_leaves': 99, 'feature_fraction': 0.9969087239121605, 'bagging_fraction': 0.7715669751528785, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=13, value=None),
#  FrozenTrial(number=37, state=1, values=[27.19963576611761], datetime_start=datetime.datetime(2025, 8, 5, 12, 36, 39, 334653), datetime_complete=datetime.datetime(2025, 8, 5, 12, 36, 57, 323218), params={'n_estimators': 357, 'learning_rate': 0.06738320429418947, 'num_leaves': 116, 'feature_fraction': 0.7238430967935205, 'bagging_fraction': 0.5008449854355049, 'min_child_samples': 21}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=37, value=None),
#  FrozenTrial(number=38, state=1, values=[27.229009635484292], datetime_start=datetime.datetime(2025, 8, 5, 12, 36, 57, 326452), datetime_complete=datetime.datetime(2025, 8, 5, 12, 37, 11, 834906), params={'n_estimators': 284, 'learning_rate': 0.09293359417382828, 'num_leaves': 106, 'feature_fraction': 0.5915977749454765, 'bagging_fraction': 0.9089207726379553, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=38, value=None),
#  FrozenTrial(number=30, state=1, values=[27.246337457265447], datetime_start=datetime.datetime(2025, 8, 5, 12, 33, 50, 653781), datetime_complete=datetime.datetime(2025, 8, 5, 12, 34, 8, 596970), params={'n_estimators': 265, 'learning_rate': 0.08024861115654236, 'num_leaves': 123, 'feature_fraction': 0.8732555778256765, 'bagging_fraction': 0.49831384089817227, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=30, value=None),
#  FrozenTrial(number=17, state=1, values=[27.26850092209443], datetime_start=datetime.datetime(2025, 8, 5, 12, 28, 48, 752114), datetime_complete=datetime.datetime(2025, 8, 5, 12, 29, 3, 953838), params={'n_estimators': 323, 'learning_rate': 0.05363493350332798, 'num_leaves': 67, 'feature_fraction': 0.6997461476449073, 'bagging_fraction': 0.8812948285685267, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=17, value=None),
#  FrozenTrial(number=1, state=1, values=[27.31308497403702], datetime_start=datetime.datetime(2025, 8, 5, 12, 23, 32, 822239), datetime_complete=datetime.datetime(2025, 8, 5, 12, 23, 52, 578267), params={'n_estimators': 492, 'learning_rate': 0.06863101570206272, 'num_leaves': 67, 'feature_fraction': 0.8349151917963709, 'bagging_fraction': 0.5835740233695073, 'min_child_samples': 45}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=1, value=None),
#  FrozenTrial(number=47, state=1, values=[27.351775728544407], datetime_start=datetime.datetime(2025, 8, 5, 12, 40, 42, 441989), datetime_complete=datetime.datetime(2025, 8, 5, 12, 41, 5, 735796), params={'n_estimators': 438, 'learning_rate': 0.0727161957733283, 'num_leaves': 114, 'feature_fraction': 0.9166411832699098, 'bagging_fraction': 0.7510774632388845, 'min_child_samples': 43}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=47, value=None),
#  FrozenTrial(number=7, state=1, values=[27.380624487373083], datetime_start=datetime.datetime(2025, 8, 5, 12, 24, 59, 934847), datetime_complete=datetime.datetime(2025, 8, 5, 12, 25, 22, 668062), params={'n_estimators': 282, 'learning_rate': 0.04160860077633204, 'num_leaves': 116, 'feature_fraction': 0.7931531513312478, 'bagging_fraction': 0.560556458234249, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=7, value=None),
#  FrozenTrial(number=6, state=1, values=[27.581687811699737], datetime_start=datetime.datetime(2025, 8, 5, 12, 24, 48, 159289), datetime_complete=datetime.datetime(2025, 8, 5, 12, 24, 59, 933586), params={'n_estimators': 387, 'learning_rate': 0.07679114297155348, 'num_leaves': 42, 'feature_fraction': 0.7549220465513488, 'bagging_fraction': 0.9059391247605363, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=6, value=None),
#  FrozenTrial(number=44, state=1, values=[27.730257470599305], datetime_start=datetime.datetime(2025, 8, 5, 12, 39, 23, 397083), datetime_complete=datetime.datetime(2025, 8, 5, 12, 39, 34, 996710), params={'n_estimators': 398, 'learning_rate': 0.09511005212136583, 'num_leaves': 53, 'feature_fraction': 0.6416349235187992, 'bagging_fraction': 0.4485497525059961, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=44, value=None),
#  FrozenTrial(number=8, state=1, values=[27.896772442116507], datetime_start=datetime.datetime(2025, 8, 5, 12, 25, 22, 671289), datetime_complete=datetime.datetime(2025, 8, 5, 12, 25, 29, 142304), params={'n_estimators': 222, 'learning_rate': 0.09298623062958333, 'num_leaves': 127, 'feature_fraction': 0.4155395660676824, 'bagging_fraction': 0.6258861632965927, 'min_child_samples': 36}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=8, value=None),
#  FrozenTrial(number=2, state=1, values=[28.0943015634397], datetime_start=datetime.datetime(2025, 8, 5, 12, 23, 52, 580414), datetime_complete=datetime.datetime(2025, 8, 5, 12, 24, 2, 644416), params={'n_estimators': 296, 'learning_rate': 0.04456943573101954, 'num_leaves': 110, 'feature_fraction': 0.5991568584622239, 'bagging_fraction': 0.48127207672921, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=2, value=None),
#  FrozenTrial(number=4, state=1, values=[28.439316732692735], datetime_start=datetime.datetime(2025, 8, 5, 12, 24, 8, 272157), datetime_complete=datetime.datetime(2025, 8, 5, 12, 24, 30, 79463), params={'n_estimators': 365, 'learning_rate': 0.017093956770675753, 'num_leaves': 122, 'feature_fraction': 0.6178639068498029, 'bagging_fraction': 0.998646746694864, 'min_child_samples': 36}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=4, value=None),
#  FrozenTrial(number=9, state=1, values=[28.877274791167082], datetime_start=datetime.datetime(2025, 8, 5, 12, 25, 29, 143560), datetime_complete=datetime.datetime(2025, 8, 5, 12, 25, 38, 485198), params={'n_estimators': 425, 'learning_rate': 0.05246888293778356, 'num_leaves': 28, 'feature_fraction': 0.7589993908042058, 'bagging_fraction': 0.4149439969108648, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=9, value=None),
#  FrozenTrial(number=5, state=1, values=[29.78769813731277], datetime_start=datetime.datetime(2025, 8, 5, 12, 24, 30, 80763), datetime_complete=datetime.datetime(2025, 8, 5, 12, 24, 48, 156191), params={'n_estimators': 383, 'learning_rate': 0.01592862012999925, 'num_leaves': 56, 'feature_fraction': 0.8376901576370752, 'bagging_fraction': 0.976091358487386, 'min_child_samples': 44}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=5, value=None),
#  FrozenTrial(number=0, state=1, values=[31.124411693121605], datetime_start=datetime.datetime(2025, 8, 5, 12, 23, 18, 879768), datetime_complete=datetime.datetime(2025, 8, 5, 12, 23, 32, 820755), params={'n_estimators': 454, 'learning_rate': 0.011784471344414117, 'num_leaves': 77, 'feature_fraction': 0.8438428602465393, 'bagging_fraction': 0.4121017716737783, 'min_child_samples': 39}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None),
#  FrozenTrial(number=3, state=1, values=[31.25381241662382], datetime_start=datetime.datetime(2025, 8, 5, 12, 24, 2, 645823), datetime_complete=datetime.datetime(2025, 8, 5, 12, 24, 8, 270907), params={'n_estimators': 373, 'learning_rate': 0.023581758700586833, 'num_leaves': 21, 'feature_fraction': 0.5742070908287209, 'bagging_fraction': 0.6246972469666091, 'min_child_samples': 33}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=3, value=None)]

In [ ]:
# desc_tg = desc_test[common_cols1]
# top_trials = sorted_tg_light[:15]
# print(len(top_trials))
# all_predictions = []
# TG_light = []
# for i, trial in enumerate(top_trials):
#     params = trial.params
#     params['objective'] = 'mae'
#     params['metric'] = 'mae'
#     params['bagging_freq'] = 1
#     params['n_jobs'] = -1
#     params['verbosity'] = -1
#     params['random_state'] = 42
#     model = Model_light(params)
#     model.train(x_tg, y_tg)
#     preds = model.evalution()
#     all_predictions.append(preds)
#     TG_light.append(model.prediction(desc_tg))
#     print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
# final_ensemble_prediction = np.mean(all_predictions, axis=0)
# final_TG_light = np.mean(TG_light, axis=0)
# print(final_ensemble_prediction)
# print(final_TG_light)

**Rg**

In [ ]:
train_cols2 = set(rg.columns) - {"Rg"}
test_cols2 = set(desc_test.columns)
common_cols2 = list(train_cols2 & test_cols2)
x_r=rg[common_cols2].copy()
y_r=rg["Rg"].copy()

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective(trial, x_r, y_r), n_trials = 50)

In [ ]:
# completed_trials_rg = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_rg = sorted(completed_trials_rg, key =lambda t: t.value)

In [ ]:
sorted_rg = [FrozenTrial(number=28, state=1, values=[1.5983000969931005], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 32, 145462), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 34, 648914), params={'n_estimators': 211, 'learning_rate': 0.05631619759056984, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=28, value=None),
 FrozenTrial(number=31, state=1, values=[1.6092192657940325], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 39, 915023), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 42, 536299), params={'n_estimators': 276, 'learning_rate': 0.06263999886029767, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=31, value=None),
 FrozenTrial(number=43, state=1, values=[1.610083501055799], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 10, 860611), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 15, 618420), params={'n_estimators': 303, 'learning_rate': 0.04918170936022993, 'max_depth': 4, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=43, value=None),
 FrozenTrial(number=48, state=1, values=[1.6129121694201958], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 32, 207127), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 38, 528968), params={'n_estimators': 356, 'learning_rate': 0.0699356170767663, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 9, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=48, value=None),
 FrozenTrial(number=49, state=1, values=[1.6130807108580252], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 38, 530062), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 45, 144663), params={'n_estimators': 387, 'learning_rate': 0.0722953646694058, 'max_depth': 5, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 10, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=49, value=None),
 FrozenTrial(number=23, state=1, values=[1.6163462109589763], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 14, 315545), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 17, 817608), params={'n_estimators': 499, 'learning_rate': 0.059721062189741284, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 8, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=23, value=None),
 FrozenTrial(number=32, state=1, values=[1.6177563882837707], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 42, 537632), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 45, 125354), params={'n_estimators': 269, 'learning_rate': 0.06606283369735265, 'max_depth': 4, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=32, value=None),
 FrozenTrial(number=6, state=1, values=[1.6207898305584865], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 43, 245153), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 51, 220660), params={'n_estimators': 298, 'learning_rate': 0.03904740789248221, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=6, value=None),
 FrozenTrial(number=22, state=1, values=[1.6283861940425322], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 8, 428908), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 14, 314615), params={'n_estimators': 497, 'learning_rate': 0.059950403840611864, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=22, value=None),
 FrozenTrial(number=46, state=1, values=[1.6284620057957988], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 24, 3682), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 28, 445951), params={'n_estimators': 286, 'learning_rate': 0.06459790094876419, 'max_depth': 4, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 10, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=46, value=None),
 FrozenTrial(number=21, state=1, values=[1.629682672889077], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 4, 529781), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 8, 427597), params={'n_estimators': 499, 'learning_rate': 0.061561810754786304, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 8, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=21, value=None),
 FrozenTrial(number=14, state=1, values=[1.6314477619864312], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 16, 986715), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 27, 910059), params={'n_estimators': 455, 'learning_rate': 0.031069399967665823, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=14, value=None),
 FrozenTrial(number=41, state=1, values=[1.63530507661895], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 5, 711337), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 8, 699260), params={'n_estimators': 284, 'learning_rate': 0.07022090243972141, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=41, value=None),
 FrozenTrial(number=12, state=1, values=[1.6391567242788125], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 10, 222655), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 13, 530439), params={'n_estimators': 471, 'learning_rate': 0.06773497586347434, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 10, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=12, value=None),
 FrozenTrial(number=25, state=1, values=[1.6405963587534538], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 23, 635442), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 27, 51053), params={'n_estimators': 484, 'learning_rate': 0.058951744640206374, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=25, value=None),
 FrozenTrial(number=38, state=1, values=[1.6416662377663513], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 57, 807829), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 0, 60371), params={'n_estimators': 344, 'learning_rate': 0.09776584202162547, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 6, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=38, value=None),
 FrozenTrial(number=26, state=1, values=[1.6420928673821176], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 27, 53510), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 29, 498878), params={'n_estimators': 375, 'learning_rate': 0.07530358007601504, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=26, value=None),
 FrozenTrial(number=16, state=1, values=[1.64323907966128], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 38, 229850), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 44, 891793), params={'n_estimators': 372, 'learning_rate': 0.034754321274713354, 'max_depth': 5, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=7, state=1, values=[1.6433362051325902], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 51, 222683), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 54, 122078), params={'n_estimators': 471, 'learning_rate': 0.0899934305247058, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=7, value=None),
 FrozenTrial(number=39, state=1, values=[1.643568746241088], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 0, 61449), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 3, 287490), params={'n_estimators': 225, 'learning_rate': 0.05104425909363964, 'max_depth': 6, 'min_child_weight': 7, 'gamma': 2, 'reg_lambda': 9, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=39, value=None),
 FrozenTrial(number=44, state=1, values=[1.6488902213807044], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 15, 619788), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 20, 909102), params={'n_estimators': 330, 'learning_rate': 0.05067201368413747, 'max_depth': 4, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=44, value=None),
 FrozenTrial(number=34, state=1, values=[1.649517052245022], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 47, 632328), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 49, 567185), params={'n_estimators': 236, 'learning_rate': 0.08178634252486204, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 2, 'reg_lambda': 6, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=34, value=None),
 FrozenTrial(number=42, state=1, values=[1.6520599129090445], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 8, 700559), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 10, 859296), params={'n_estimators': 257, 'learning_rate': 0.05739617727032797, 'max_depth': 4, 'min_child_weight': 2, 'gamma': 3, 'reg_lambda': 9, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=42, value=None),
 FrozenTrial(number=4, state=1, values=[1.6529692449529783], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 34, 338775), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 39, 64314), params={'n_estimators': 488, 'learning_rate': 0.04717183056375647, 'max_depth': 9, 'min_child_weight': 10, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=4, value=None),
 FrozenTrial(number=36, state=1, values=[1.6529872101515655], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 51, 611402), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 55, 573310), params={'n_estimators': 319, 'learning_rate': 0.06306425698620256, 'max_depth': 4, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 7, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=36, value=None),
 FrozenTrial(number=33, state=1, values=[1.65476764430805], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 45, 126650), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 47, 629325), params={'n_estimators': 272, 'learning_rate': 0.0733139423921585, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=33, value=None),
 FrozenTrial(number=45, state=1, values=[1.6576208992798644], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 20, 912322), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 24, 168), params={'n_estimators': 307, 'learning_rate': 0.043168397168831994, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 8, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=45, value=None),
 FrozenTrial(number=0, state=1, values=[1.6600060473782474], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 21, 640272), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 23, 971843), params={'n_estimators': 299, 'learning_rate': 0.06490481046066415, 'max_depth': 4, 'min_child_weight': 10, 'gamma': 3, 'reg_lambda': 6, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=5, state=1, values=[1.6601483976739153], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 39, 65437), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 43, 244023), params={'n_estimators': 291, 'learning_rate': 0.03956597876293279, 'max_depth': 5, 'min_child_weight': 1, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=5, value=None),
 FrozenTrial(number=27, state=1, values=[1.660425242927246], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 29, 501778), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 32, 143065), params={'n_estimators': 267, 'learning_rate': 0.04565455518374763, 'max_depth': 3, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=27, value=None),
 FrozenTrial(number=40, state=1, values=[1.6628648614227044], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 3, 288884), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 5, 709879), params={'n_estimators': 309, 'learning_rate': 0.08044780268165469, 'max_depth': 5, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=13, state=1, values=[1.6669891453786558], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 13, 532720), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 16, 984296), params={'n_estimators': 388, 'learning_rate': 0.06997936591122582, 'max_depth': 6, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 10, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=13, value=None),
 FrozenTrial(number=18, state=1, values=[1.6715721410139577], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 49, 39836), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 56, 472561), params={'n_estimators': 244, 'learning_rate': 0.02297755697892824, 'max_depth': 6, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 6, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=18, value=None),
 FrozenTrial(number=8, state=1, values=[1.6716773575577997], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 54, 124926), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 57, 429569), params={'n_estimators': 407, 'learning_rate': 0.05303828071270044, 'max_depth': 8, 'min_child_weight': 10, 'gamma': 3, 'reg_lambda': 9, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=11, state=1, values=[1.6725014434381154], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 6, 316376), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 10, 219641), params={'n_estimators': 415, 'learning_rate': 0.028760722930807503, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=11, value=None),
 FrozenTrial(number=1, state=1, values=[1.6743087330240087], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 23, 973060), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 29, 75194), params={'n_estimators': 347, 'learning_rate': 0.05223186997608101, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=1, value=None),
 FrozenTrial(number=24, state=1, values=[1.6745396927484277], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 17, 818673), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 23, 632595), params={'n_estimators': 443, 'learning_rate': 0.04657054299826014, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=29, state=1, values=[1.6822170797490907], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 34, 650166), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 37, 924331), params={'n_estimators': 212, 'learning_rate': 0.03871060923049341, 'max_depth': 6, 'min_child_weight': 6, 'gamma': 3, 'reg_lambda': 6, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=29, value=None),
 FrozenTrial(number=30, state=1, values=[1.6835007927838979], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 37, 925631), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 39, 913677), params={'n_estimators': 223, 'learning_rate': 0.05331685473603254, 'max_depth': 3, 'min_child_weight': 5, 'gamma': 2, 'reg_lambda': 5, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=30, value=None),
 FrozenTrial(number=37, state=1, values=[1.6836317066428481], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 55, 576074), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 57, 806561), params={'n_estimators': 272, 'learning_rate': 0.05758707234928234, 'max_depth': 3, 'min_child_weight': 1, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=37, value=None),
 FrozenTrial(number=35, state=1, values=[1.6839740524272737], datetime_start=datetime.datetime(2025, 8, 5, 13, 19, 49, 569504), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 51, 610095), params={'n_estimators': 200, 'learning_rate': 0.06620946651205806, 'max_depth': 3, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=35, value=None),
 FrozenTrial(number=47, state=1, values=[1.684922086891673], datetime_start=datetime.datetime(2025, 8, 5, 13, 20, 28, 447053), datetime_complete=datetime.datetime(2025, 8, 5, 13, 20, 32, 206044), params={'n_estimators': 233, 'learning_rate': 0.05497121648263341, 'max_depth': 8, 'min_child_weight': 3, 'gamma': 2, 'reg_lambda': 8, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=47, value=None),
 FrozenTrial(number=17, state=1, values=[1.6860993649448996], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 44, 893529), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 49, 38729), params={'n_estimators': 441, 'learning_rate': 0.012960551889603718, 'max_depth': 3, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=17, value=None),
 FrozenTrial(number=2, state=1, values=[1.6861650061154843], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 29, 76443), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 31, 316552), params={'n_estimators': 250, 'learning_rate': 0.09438463720977017, 'max_depth': 6, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 10, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=2, value=None),
 FrozenTrial(number=19, state=1, values=[1.686258010884325], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 56, 473603), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 59, 931070), params={'n_estimators': 316, 'learning_rate': 0.039681621803760475, 'max_depth': 10, 'min_child_weight': 7, 'gamma': 5, 'reg_lambda': 8, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=19, value=None),
 FrozenTrial(number=3, state=1, values=[1.6894903314118077], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 31, 317794), datetime_complete=datetime.datetime(2025, 8, 5, 13, 17, 34, 337547), params={'n_estimators': 419, 'learning_rate': 0.07746360020285639, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=3, value=None),
 FrozenTrial(number=15, state=1, values=[1.7021277865720474], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 27, 911938), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 38, 228748), params={'n_estimators': 292, 'learning_rate': 0.026084668644012654, 'max_depth': 7, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=15, value=None),
 FrozenTrial(number=20, state=1, values=[1.7041045786616131], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 59, 932006), datetime_complete=datetime.datetime(2025, 8, 5, 13, 19, 4, 528699), params={'n_estimators': 254, 'learning_rate': 0.020116557671833898, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 4, 'reg_lambda': 3, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=20, value=None),
 FrozenTrial(number=10, state=1, values=[1.7646073716225228], datetime_start=datetime.datetime(2025, 8, 5, 13, 18, 0, 387619), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 6, 315284), params={'n_estimators': 208, 'learning_rate': 0.012913760508730075, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 5, 'reg_lambda': 7, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=10, value=None),
 FrozenTrial(number=9, state=1, values=[1.7776035840954094], datetime_start=datetime.datetime(2025, 8, 5, 13, 17, 57, 430871), datetime_complete=datetime.datetime(2025, 8, 5, 13, 18, 0, 384812), params={'n_estimators': 332, 'learning_rate': 0.08468608823039332, 'max_depth': 9, 'min_child_weight': 1, 'gamma': 4, 'reg_lambda': 3, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=9, value=None)]


In [ ]:
desc_rg = desc_test[common_cols2]
top_trials = sorted_rg[:10]
all_predictions = []
RG = []
for i, trial in enumerate(top_trials):
    params = trial.params
    params['random_state'] = 42+i
    model = Model_XG(params)
    model.train(x_r, y_r)
    preds = model.evalution()
    all_predictions.append(preds)
    RG.append(model.prediction(desc_rg))
    print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
final_ensemble_prediction = np.mean(all_predictions, axis=0)
final_RG = np.mean(RG, axis=0)
print(final_ensemble_prediction)
print(final_RG)

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective_light(trial, x_r, y_r), n_trials = 50)

In [ ]:
# completed_trials_rg_light = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_rg_light = sorted(completed_trials_rg_light, key =lambda t: t.value)

In [ ]:
# sorted_rg_light = [FrozenTrial(number=34, state=1, values=[1.6991912502461821], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 11, 300528), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 13, 895065), params={'n_estimators': 316, 'learning_rate': 0.027490539049437806, 'num_leaves': 110, 'feature_fraction': 0.9383434345375766, 'bagging_fraction': 0.7267963893294467, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=34, value=None),
#  FrozenTrial(number=10, state=1, values=[1.7071734455319265], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 15, 544550), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 18, 964490), params={'n_estimators': 356, 'learning_rate': 0.014802643483303402, 'num_leaves': 127, 'feature_fraction': 0.8038048995451529, 'bagging_fraction': 0.738009650863726, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=10, value=None),
#  FrozenTrial(number=36, state=1, values=[1.7140602201045732], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 15, 320757), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 16, 687566), params={'n_estimators': 309, 'learning_rate': 0.05573089377143497, 'num_leaves': 101, 'feature_fraction': 0.941668314889348, 'bagging_fraction': 0.8692926789322226, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=36, value=None),
#  FrozenTrial(number=35, state=1, values=[1.7155876107439427], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 13, 896820), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 15, 319013), params={'n_estimators': 309, 'learning_rate': 0.05506091750369406, 'num_leaves': 101, 'feature_fraction': 0.926497311032926, 'bagging_fraction': 0.866034219224957, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=35, value=None),
#  FrozenTrial(number=39, state=1, values=[1.72179456823373], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 19, 49697), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 20, 875526), params={'n_estimators': 330, 'learning_rate': 0.0636533953142025, 'num_leaves': 73, 'feature_fraction': 0.923952027096838, 'bagging_fraction': 0.9994890257514444, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=39, value=None),
#  FrozenTrial(number=27, state=1, values=[1.7221244950299348], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 56, 95049), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 58, 363455), params={'n_estimators': 265, 'learning_rate': 0.0410639411533182, 'num_leaves': 107, 'feature_fraction': 0.8917449720634744, 'bagging_fraction': 0.8054611858228325, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=27, value=None),
#  FrozenTrial(number=12, state=1, values=[1.7231661828928393], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 22, 567148), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 25, 895901), params={'n_estimators': 368, 'learning_rate': 0.010143929997309559, 'num_leaves': 127, 'feature_fraction': 0.7713999964050986, 'bagging_fraction': 0.7517420047033052, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=12, value=None),
#  FrozenTrial(number=31, state=1, values=[1.7260260443882849], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 4, 53475), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 7, 416110), params={'n_estimators': 354, 'learning_rate': 0.025091225319391654, 'num_leaves': 114, 'feature_fraction': 0.7516103560532315, 'bagging_fraction': 0.7337729402155312, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=31, value=None),
#  FrozenTrial(number=11, state=1, values=[1.7266779152435106], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 18, 965666), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 22, 565917), params={'n_estimators': 339, 'learning_rate': 0.012193264805616146, 'num_leaves': 118, 'feature_fraction': 0.7984590125728643, 'bagging_fraction': 0.7329150304031331, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=11, value=None),
#  FrozenTrial(number=19, state=1, values=[1.727226697716587], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 40, 991244), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 42, 660889), params={'n_estimators': 402, 'learning_rate': 0.04770744500816883, 'num_leaves': 55, 'feature_fraction': 0.7378222224677196, 'bagging_fraction': 0.901144874161994, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=19, value=None),
#  FrozenTrial(number=24, state=1, values=[1.7296531211323418], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 49, 428341), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 53, 140868), params={'n_estimators': 392, 'learning_rate': 0.017475410915672507, 'num_leaves': 121, 'feature_fraction': 0.8093858765818335, 'bagging_fraction': 0.7295645524385073, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=24, value=None),
#  FrozenTrial(number=22, state=1, values=[1.7340688864344158], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 45, 871366), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 47, 720755), params={'n_estimators': 351, 'learning_rate': 0.01657048816678606, 'num_leaves': 118, 'feature_fraction': 0.7250922026298662, 'bagging_fraction': 0.6698605614992047, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=22, value=None),
#  FrozenTrial(number=28, state=1, values=[1.7368064216059302], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 58, 364701), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 1, 518144), params={'n_estimators': 235, 'learning_rate': 0.043881514308692485, 'num_leaves': 106, 'feature_fraction': 0.9076686514739221, 'bagging_fraction': 0.9221921481691562, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=28, value=None),
#  FrozenTrial(number=21, state=1, values=[1.7429784387823515], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 43, 294333), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 45, 870199), params={'n_estimators': 337, 'learning_rate': 0.010621320184061912, 'num_leaves': 119, 'feature_fraction': 0.7960296462205279, 'bagging_fraction': 0.731418449888098, 'min_child_samples': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=21, value=None),
#  FrozenTrial(number=13, state=1, values=[1.743947550714222], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 25, 896918), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 30, 997767), params={'n_estimators': 370, 'learning_rate': 0.011572754643935819, 'num_leaves': 126, 'feature_fraction': 0.7631596791018888, 'bagging_fraction': 0.7730918254943221, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=13, value=None),
#  FrozenTrial(number=0, state=1, values=[1.7440935871445946], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 7, 833148), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 9, 46059), params={'n_estimators': 327, 'learning_rate': 0.03511437227913545, 'num_leaves': 70, 'feature_fraction': 0.9927356803660536, 'bagging_fraction': 0.6360600512723762, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None),
#  FrozenTrial(number=18, state=1, values=[1.745525527597989], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 38, 517355), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 40, 990113), params={'n_estimators': 322, 'learning_rate': 0.026267358800678632, 'num_leaves': 130, 'feature_fraction': 0.8716164218899501, 'bagging_fraction': 0.7795569717197005, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=18, value=None),
#  FrozenTrial(number=23, state=1, values=[1.748617796563188], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 47, 721937), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 49, 427296), params={'n_estimators': 350, 'learning_rate': 0.030558287099841007, 'num_leaves': 104, 'feature_fraction': 0.9037407992515176, 'bagging_fraction': 0.7985481194038806, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=23, value=None),
#  FrozenTrial(number=29, state=1, values=[1.7503444899323934], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 1, 519455), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 2, 431653), params={'n_estimators': 267, 'learning_rate': 0.06441611143008243, 'num_leaves': 68, 'feature_fraction': 0.8527527582839312, 'bagging_fraction': 0.7998283932869039, 'min_child_samples': 21}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=29, value=None),
#  FrozenTrial(number=4, state=1, values=[1.7559786568470643], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 11, 90899), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 12, 243883), params={'n_estimators': 398, 'learning_rate': 0.040493215018551514, 'num_leaves': 21, 'feature_fraction': 0.47117032928895053, 'bagging_fraction': 0.8354357569893369, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=4, value=None),
#  FrozenTrial(number=32, state=1, values=[1.7576952717662244], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 7, 417297), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 9, 409053), params={'n_estimators': 358, 'learning_rate': 0.023563171962586366, 'num_leaves': 112, 'feature_fraction': 0.7423586480216344, 'bagging_fraction': 0.646375037631802, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=32, value=None),
#  FrozenTrial(number=41, state=1, values=[1.761924774562853], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 21, 620453), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 22, 993369), params={'n_estimators': 333, 'learning_rate': 0.0597699934534375, 'num_leaves': 68, 'feature_fraction': 0.9323775047452091, 'bagging_fraction': 0.9897621441072404, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=41, value=None),
#  FrozenTrial(number=37, state=1, values=[1.7631564664292296], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 16, 689261), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 17, 718807), params={'n_estimators': 306, 'learning_rate': 0.05624799998368554, 'num_leaves': 99, 'feature_fraction': 0.9357040958961274, 'bagging_fraction': 0.8646912295458532, 'min_child_samples': 28}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=37, value=None),
#  FrozenTrial(number=48, state=1, values=[1.7684110796407444], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 29, 784371), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 31, 311665), params={'n_estimators': 328, 'learning_rate': 0.0681079665685561, 'num_leaves': 42, 'feature_fraction': 0.9765497375685356, 'bagging_fraction': 0.8260731271429703, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=48, value=None),
#  FrozenTrial(number=43, state=1, values=[1.769905882144536], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 24, 581081), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 25, 926956), params={'n_estimators': 278, 'learning_rate': 0.05087148258726451, 'num_leaves': 80, 'feature_fraction': 0.9962003963266329, 'bagging_fraction': 0.9560657864089768, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=43, value=None),
#  FrozenTrial(number=8, state=1, values=[1.7719401045430971], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 13, 737604), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 15, 163626), params={'n_estimators': 437, 'learning_rate': 0.07994068337487119, 'num_leaves': 96, 'feature_fraction': 0.9815908134929737, 'bagging_fraction': 0.6034517046567439, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=8, value=None),
#  FrozenTrial(number=25, state=1, values=[1.7740493935318715], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 53, 142196), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 55, 594935), params={'n_estimators': 428, 'learning_rate': 0.03016174324668809, 'num_leaves': 130, 'feature_fraction': 0.6518943646876694, 'bagging_fraction': 0.8779168729581963, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=25, value=None),
#  FrozenTrial(number=15, state=1, values=[1.7763104004589945], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 34, 559030), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 36, 55985), params={'n_estimators': 375, 'learning_rate': 0.020643893912104673, 'num_leaves': 127, 'feature_fraction': 0.6789786137996763, 'bagging_fraction': 0.6994649468704901, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=15, value=None),
#  FrozenTrial(number=17, state=1, values=[1.7763934110403101], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 37, 641158), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 38, 516107), params={'n_estimators': 367, 'learning_rate': 0.010705943760059915, 'num_leaves': 86, 'feature_fraction': 0.8301686362297708, 'bagging_fraction': 0.6936622072645098, 'min_child_samples': 24}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=17, value=None),
#  FrozenTrial(number=16, state=1, values=[1.7769212136817643], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 36, 57021), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 37, 639951), params={'n_estimators': 290, 'learning_rate': 0.048558372355366454, 'num_leaves': 108, 'feature_fraction': 0.6927263919553736, 'bagging_fraction': 0.8538608717633771, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=16, value=None),
#  FrozenTrial(number=46, state=1, values=[1.7786816839227917], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 27, 487144), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 29, 302519), params={'n_estimators': 299, 'learning_rate': 0.060521857499343176, 'num_leaves': 65, 'feature_fraction': 0.9609082178791299, 'bagging_fraction': 0.8511959697820635, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=46, value=None),
#  FrozenTrial(number=38, state=1, values=[1.7793698521825654], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 17, 721068), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 19, 47466), params={'n_estimators': 287, 'learning_rate': 0.0563282258916319, 'num_leaves': 87, 'feature_fraction': 0.9937168069653137, 'bagging_fraction': 0.9553223007378769, 'min_child_samples': 20}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=38, value=None),
#  FrozenTrial(number=14, state=1, values=[1.780424549019126], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 30, 998928), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 34, 557813), params={'n_estimators': 500, 'learning_rate': 0.02266366390923507, 'num_leaves': 115, 'feature_fraction': 0.8478549039208647, 'bagging_fraction': 0.9997325059414409, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=14, value=None),
#  FrozenTrial(number=42, state=1, values=[1.7836105815270638], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 22, 995337), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 24, 579894), params={'n_estimators': 299, 'learning_rate': 0.07039971833119985, 'num_leaves': 76, 'feature_fraction': 0.9339612451992443, 'bagging_fraction': 0.906571903750202, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=42, value=None),
#  FrozenTrial(number=33, state=1, values=[1.78980941813716], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 9, 410185), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 11, 298847), params={'n_estimators': 381, 'learning_rate': 0.032123861021690334, 'num_leaves': 124, 'feature_fraction': 0.8793914208621006, 'bagging_fraction': 0.8244193010582681, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=33, value=None),
#  FrozenTrial(number=20, state=1, values=[1.7930034186073365], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 42, 662061), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 43, 293152), params={'n_estimators': 265, 'learning_rate': 0.0203689739989817, 'num_leaves': 112, 'feature_fraction': 0.626926778061694, 'bagging_fraction': 0.7569462777841424, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=20, value=None),
#  FrozenTrial(number=1, state=1, values=[1.797179944528553], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 9, 47430), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 10, 211663), params={'n_estimators': 417, 'learning_rate': 0.05446533169136394, 'num_leaves': 37, 'feature_fraction': 0.97558448077926, 'bagging_fraction': 0.9357252759019173, 'min_child_samples': 46}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=1, value=None),
#  FrozenTrial(number=45, state=1, values=[1.8005243350536564], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 26, 398446), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 27, 485413), params={'n_estimators': 312, 'learning_rate': 0.08782614657144486, 'num_leaves': 98, 'feature_fraction': 0.8638061320868603, 'bagging_fraction': 0.8805261492801607, 'min_child_samples': 26}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=45, value=None),
#  FrozenTrial(number=26, state=1, values=[1.8013623343089584], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 55, 596116), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 56, 93819), params={'n_estimators': 314, 'learning_rate': 0.0170215572634395, 'num_leaves': 93, 'feature_fraction': 0.5563184450354672, 'bagging_fraction': 0.6782076556665713, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=26, value=None),
#  FrozenTrial(number=7, state=1, values=[1.8027516529520848], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 13, 364368), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 13, 736377), params={'n_estimators': 228, 'learning_rate': 0.09190609197543398, 'num_leaves': 22, 'feature_fraction': 0.9520784829505083, 'bagging_fraction': 0.42633176847133897, 'min_child_samples': 26}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=7, value=None),
#  FrozenTrial(number=30, state=1, values=[1.8028876385713242], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 2, 432929), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 4, 52359), params={'n_estimators': 250, 'learning_rate': 0.03640723396233402, 'num_leaves': 85, 'feature_fraction': 0.415240031891127, 'bagging_fraction': 0.9581640870545376, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=30, value=None),
#  FrozenTrial(number=3, state=1, values=[1.807888153072019], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 10, 478935), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 11, 89684), params={'n_estimators': 284, 'learning_rate': 0.061881627173093735, 'num_leaves': 79, 'feature_fraction': 0.9029058303433347, 'bagging_fraction': 0.8320104824763085, 'min_child_samples': 45}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=3, value=None),
#  FrozenTrial(number=49, state=1, values=[1.8144450058011317], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 31, 312921), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 33, 852925), params={'n_estimators': 207, 'learning_rate': 0.04373388056307681, 'num_leaves': 74, 'feature_fraction': 0.8431189838933861, 'bagging_fraction': 0.5612474096345546, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=49, value=None),
#  FrozenTrial(number=40, state=1, values=[1.8237417744946869], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 20, 877339), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 21, 618640), params={'n_estimators': 318, 'learning_rate': 0.06692082596249438, 'num_leaves': 53, 'feature_fraction': 0.9585973942027279, 'bagging_fraction': 0.42196681670926034, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=40, value=None),
#  FrozenTrial(number=5, state=1, values=[1.8254822005583007], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 12, 245426), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 12, 611741), params={'n_estimators': 224, 'learning_rate': 0.03680549363307941, 'num_leaves': 32, 'feature_fraction': 0.9651867096950184, 'bagging_fraction': 0.5658215240008636, 'min_child_samples': 38}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=5, value=None),
#  FrozenTrial(number=44, state=1, values=[1.8517338563457382], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 25, 928924), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 26, 395964), params={'n_estimators': 330, 'learning_rate': 0.07646668577323164, 'num_leaves': 92, 'feature_fraction': 0.9183702596167713, 'bagging_fraction': 0.5966314364514707, 'min_child_samples': 44}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=44, value=None),
#  FrozenTrial(number=6, state=1, values=[1.8528942228111112], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 12, 613065), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 13, 363060), params={'n_estimators': 463, 'learning_rate': 0.08109705088857819, 'num_leaves': 94, 'feature_fraction': 0.7754452381136451, 'bagging_fraction': 0.6261874053526599, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=6, value=None),
#  FrozenTrial(number=9, state=1, values=[1.8611979797827092], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 15, 165050), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 15, 543281), params={'n_estimators': 302, 'learning_rate': 0.07228381224385089, 'num_leaves': 102, 'feature_fraction': 0.621506578345792, 'bagging_fraction': 0.5264035964352668, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=9, value=None),
#  FrozenTrial(number=47, state=1, values=[1.8663630836652063], datetime_start=datetime.datetime(2025, 8, 5, 13, 40, 29, 303728), datetime_complete=datetime.datetime(2025, 8, 5, 13, 40, 29, 782666), params={'n_estimators': 338, 'learning_rate': 0.05409941099047108, 'num_leaves': 81, 'feature_fraction': 0.823677677388793, 'bagging_fraction': 0.4674468162766462, 'min_child_samples': 30}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=47, value=None),
#  FrozenTrial(number=2, state=1, values=[1.8997349252599935], datetime_start=datetime.datetime(2025, 8, 5, 13, 39, 10, 213385), datetime_complete=datetime.datetime(2025, 8, 5, 13, 39, 10, 477447), params={'n_estimators': 200, 'learning_rate': 0.028597367478513165, 'num_leaves': 60, 'feature_fraction': 0.5619193932440255, 'bagging_fraction': 0.5238863162609088, 'min_child_samples': 49}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=2, value=None)]

In [ ]:
# desc_rg = desc_test[common_cols2]
# top_trials = sorted_rg_light[:15]
# all_predictions = []
# RG_light = []
# for i, trial in enumerate(top_trials):
#     params = trial.params
#     params['objective'] = 'mae'
#     params['metric'] = 'mae'
#     params['bagging_freq'] = 1
#     params['n_jobs'] = -1
#     params['verbosity'] = -1
#     params['random_state'] = 42
#     model = Model_light(params)
#     model.train(x_r, y_r)
#     preds = model.evalution()
#     all_predictions.append(preds)
#     RG_light.append(model.prediction(desc_rg))
#     print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
# final_ensemble_prediction = np.mean(all_predictions, axis=0)
# final_RG_light = np.mean(RG_light, axis=0)
# print(final_ensemble_prediction)
# print(final_RG_light)

**FFV**

In [ ]:
train_cols3 = set(ffv.columns) - {"FFV"}
test_cols3 = set(desc_test.columns)
common_cols3 = list(train_cols3 & test_cols3)
x_f=ffv[common_cols3].copy()
y_f=ffv["FFV"].copy()

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective(trial, x_f, y_f), n_trials = 50)

In [ ]:
# completed_trials_ffv = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_ffv = sorted(completed_trials_ffv, key =lambda t: t.value)

In [ ]:
sorted_ffv = [FrozenTrial(number=47, state=1, values=[0.006740145885478], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 28, 973376), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 37, 647267), params={'n_estimators': 210, 'learning_rate': 0.09216145528904694, 'max_depth': 5, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=47, value=None),
 FrozenTrial(number=34, state=1, values=[0.007809133545780604], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 15, 571666), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 24, 481578), params={'n_estimators': 240, 'learning_rate': 0.0872244551683873, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=34, value=None),
 FrozenTrial(number=42, state=1, values=[0.007875201839476506], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 2, 420357), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 10, 121401), params={'n_estimators': 209, 'learning_rate': 0.09439284104050652, 'max_depth': 5, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=42, value=None),
 FrozenTrial(number=41, state=1, values=[0.007944415756259076], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 55, 113567), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 2, 418972), params={'n_estimators': 200, 'learning_rate': 0.09303260081516256, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=41, value=None),
 FrozenTrial(number=40, state=1, values=[0.007971824423713516], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 47, 123506), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 55, 112459), params={'n_estimators': 201, 'learning_rate': 0.08422233142298839, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=43, state=1, values=[0.008158750744424877], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 10, 122582), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 16, 314933), params={'n_estimators': 213, 'learning_rate': 0.09404685141422515, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=43, value=None),
 FrozenTrial(number=46, state=1, values=[0.008445478969781093], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 23, 538993), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 28, 972273), params={'n_estimators': 238, 'learning_rate': 0.08205543927570254, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=46, value=None),
 FrozenTrial(number=33, state=1, values=[0.008845629595586161], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 6, 711178), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 15, 570580), params={'n_estimators': 316, 'learning_rate': 0.08612209866898675, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=33, value=None),
 FrozenTrial(number=38, state=1, values=[0.00892575478120151], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 37, 3350), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 43, 563357), params={'n_estimators': 201, 'learning_rate': 0.08828277729923044, 'max_depth': 6, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=38, value=None),
 FrozenTrial(number=36, state=1, values=[0.008979623634505928], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 27, 595283), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 33, 926586), params={'n_estimators': 254, 'learning_rate': 0.08711823835807495, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=36, value=None),
 FrozenTrial(number=25, state=1, values=[0.009903915420556452], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 17, 93992), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 26, 705429), params={'n_estimators': 372, 'learning_rate': 0.06075756711218623, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=25, value=None),
 FrozenTrial(number=31, state=1, values=[0.009905167509048962], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 54, 973838), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 3, 166915), params={'n_estimators': 314, 'learning_rate': 0.0705878070567301, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=31, value=None),
 FrozenTrial(number=30, state=1, values=[0.009942452400907147], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 47, 8987), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 54, 972790), params={'n_estimators': 315, 'learning_rate': 0.07036212110975945, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=30, value=None),
 FrozenTrial(number=24, state=1, values=[0.010393230844931945], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 6, 410672), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 17, 92833), params={'n_estimators': 368, 'learning_rate': 0.06166052890957189, 'max_depth': 10, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=28, state=1, values=[0.010464307174008133], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 34, 328772), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 42, 64197), params={'n_estimators': 367, 'learning_rate': 0.0773465509925083, 'max_depth': 9, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=28, value=None),
 FrozenTrial(number=17, state=1, values=[0.010894344028566615], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 10, 547062), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 25, 367168), params={'n_estimators': 394, 'learning_rate': 0.027010631509113044, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=17, value=None),
 FrozenTrial(number=21, state=1, values=[0.011285889053283683], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 40, 572579), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 51, 716742), params={'n_estimators': 447, 'learning_rate': 0.04012990356851454, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=21, value=None),
 FrozenTrial(number=22, state=1, values=[0.011308949418721015], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 51, 717846), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 1, 533497), params={'n_estimators': 460, 'learning_rate': 0.051467635192094865, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=22, value=None),
 FrozenTrial(number=14, state=1, values=[0.011373286018807632], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 49, 169823), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 59, 655080), params={'n_estimators': 442, 'learning_rate': 0.03663806735548465, 'max_depth': 8, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=14, value=None),
 FrozenTrial(number=1, state=1, values=[0.011662835635166399], datetime_start=datetime.datetime(2025, 8, 5, 13, 31, 35, 963101), datetime_complete=datetime.datetime(2025, 8, 5, 13, 31, 46, 930304), params={'n_estimators': 432, 'learning_rate': 0.0386964511019681, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=1, value=None),
 FrozenTrial(number=12, state=1, values=[0.011790018293870306], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 37, 781631), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 46, 300945), params={'n_estimators': 334, 'learning_rate': 0.04044765080422023, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=12, value=None),
 FrozenTrial(number=3, state=1, values=[0.012110195161391227], datetime_start=datetime.datetime(2025, 8, 5, 13, 31, 50, 950537), datetime_complete=datetime.datetime(2025, 8, 5, 13, 31, 57, 44342), params={'n_estimators': 334, 'learning_rate': 0.08984054859854582, 'max_depth': 9, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=3, value=None),
 FrozenTrial(number=8, state=1, values=[0.01239236806551186], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 13, 134728), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 20, 190426), params={'n_estimators': 344, 'learning_rate': 0.062367700995803244, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=11, state=1, values=[0.0123974790297629], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 29, 877746), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 37, 780511), params={'n_estimators': 425, 'learning_rate': 0.09991056418761918, 'max_depth': 9, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 10, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=11, value=None),
 FrozenTrial(number=0, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 31, 31, 429720), datetime_complete=datetime.datetime(2025, 8, 5, 13, 31, 35, 962010), params={'n_estimators': 387, 'learning_rate': 0.02255995552887231, 'max_depth': 4, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 8, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=2, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 31, 46, 931367), datetime_complete=datetime.datetime(2025, 8, 5, 13, 31, 50, 949388), params={'n_estimators': 353, 'learning_rate': 0.022233442233477185, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=2, value=None),
 FrozenTrial(number=4, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 31, 57, 46149), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 2, 258191), params={'n_estimators': 478, 'learning_rate': 0.010108295587941612, 'max_depth': 4, 'min_child_weight': 1, 'gamma': 5, 'reg_lambda': 1, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=4, value=None),
 FrozenTrial(number=5, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 2, 259628), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 6, 628253), params={'n_estimators': 278, 'learning_rate': 0.08152476222950475, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 2, 'reg_lambda': 7, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=5, value=None),
 FrozenTrial(number=6, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 6, 629535), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 10, 4934), params={'n_estimators': 288, 'learning_rate': 0.07019559152927887, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 9, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=6, value=None),
 FrozenTrial(number=7, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 10, 6091), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 13, 133663), params={'n_estimators': 268, 'learning_rate': 0.06378049652318188, 'max_depth': 4, 'min_child_weight': 9, 'gamma': 2, 'reg_lambda': 7, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=7, value=None),
 FrozenTrial(number=9, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 20, 191612), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 24, 500934), params={'n_estimators': 398, 'learning_rate': 0.010640650337691895, 'max_depth': 10, 'min_child_weight': 3, 'gamma': 3, 'reg_lambda': 4, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=9, value=None),
 FrozenTrial(number=10, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 24, 502052), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 29, 876560), params={'n_estimators': 500, 'learning_rate': 0.043450201538222114, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 3, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=10, value=None),
 FrozenTrial(number=13, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 46, 302017), datetime_complete=datetime.datetime(2025, 8, 5, 13, 32, 49, 168664), params={'n_estimators': 225, 'learning_rate': 0.0465386907645061, 'max_depth': 7, 'min_child_weight': 10, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=13, value=None),
 FrozenTrial(number=15, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 32, 59, 656234), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 4, 497051), params={'n_estimators': 453, 'learning_rate': 0.032927486830945644, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 5, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=15, value=None),
 FrozenTrial(number=16, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 4, 498257), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 10, 545961), params={'n_estimators': 431, 'learning_rate': 0.05039339193885462, 'max_depth': 8, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 3, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=18, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 25, 368255), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 29, 695077), params={'n_estimators': 382, 'learning_rate': 0.027682659605514866, 'max_depth': 6, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 6, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=18, value=None),
 FrozenTrial(number=19, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 29, 696168), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 34, 700698), params={'n_estimators': 475, 'learning_rate': 0.031085621110708447, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 5, 'reg_lambda': 7, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=19, value=None),
 FrozenTrial(number=20, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 33, 34, 701751), datetime_complete=datetime.datetime(2025, 8, 5, 13, 33, 40, 571466), params={'n_estimators': 409, 'learning_rate': 0.01882975626111262, 'max_depth': 7, 'min_child_weight': 6, 'gamma': 4, 'reg_lambda': 6, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=20, value=None),
 FrozenTrial(number=23, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 1, 534604), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 6, 409562), params={'n_estimators': 458, 'learning_rate': 0.05358543649723018, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=23, value=None),
 FrozenTrial(number=26, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 26, 706587), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 30, 811160), params={'n_estimators': 369, 'learning_rate': 0.07243262807197029, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=26, value=None),
 FrozenTrial(number=27, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 30, 812237), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 34, 323621), params={'n_estimators': 312, 'learning_rate': 0.060727265272695466, 'max_depth': 3, 'min_child_weight': 2, 'gamma': 2, 'reg_lambda': 2, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=27, value=None),
 FrozenTrial(number=29, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 34, 42, 65363), datetime_complete=datetime.datetime(2025, 8, 5, 13, 34, 47, 7903), params={'n_estimators': 369, 'learning_rate': 0.07880413449204693, 'max_depth': 9, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=29, value=None),
 FrozenTrial(number=32, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 3, 167955), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 6, 710088), params={'n_estimators': 307, 'learning_rate': 0.06806394479172438, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=32, value=None),
 FrozenTrial(number=35, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 24, 482939), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 27, 594180), params={'n_estimators': 238, 'learning_rate': 0.0887947578069141, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=35, value=None),
 FrozenTrial(number=37, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 33, 927658), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 37, 2186), params={'n_estimators': 254, 'learning_rate': 0.09948831323408328, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=37, value=None),
 FrozenTrial(number=39, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 35, 43, 564469), datetime_complete=datetime.datetime(2025, 8, 5, 13, 35, 47, 122387), params={'n_estimators': 202, 'learning_rate': 0.08822157276105301, 'max_depth': 6, 'min_child_weight': 3, 'gamma': 2, 'reg_lambda': 3, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=39, value=None),
 FrozenTrial(number=44, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 16, 316154), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 20, 644202), params={'n_estimators': 219, 'learning_rate': 0.09603216921181636, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=44, value=None),
 FrozenTrial(number=45, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 20, 645359), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 23, 537918), params={'n_estimators': 223, 'learning_rate': 0.09417259135353671, 'max_depth': 4, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=45, value=None),
 FrozenTrial(number=48, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 37, 648343), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 40, 750040), params={'n_estimators': 240, 'learning_rate': 0.0827375634373867, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=48, value=None),
 FrozenTrial(number=49, state=1, values=[0.02094338035005327], datetime_start=datetime.datetime(2025, 8, 5, 13, 36, 40, 751514), datetime_complete=datetime.datetime(2025, 8, 5, 13, 36, 44, 23642), params={'n_estimators': 273, 'learning_rate': 0.09339239088388855, 'max_depth': 5, 'min_child_weight': 4, 'gamma': 3, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=49, value=None)]

In [ ]:
desc_ffv = desc_test[common_cols3]
top_trials = sorted_ffv[:10]
all_predictions = []
FFV = []
for i, trial in enumerate(top_trials):
    params = trial.params
    params['random_state'] = 42+i
    model = Model_XG(params)
    model.train(x_f, y_f)
    preds = model.evalution()
    all_predictions.append(preds)
    FFV.append(model.prediction(desc_ffv))
    print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
final_ensemble_prediction = np.mean(all_predictions, axis=0)
final_FFV = np.mean(FFV, axis=0)
print(final_ensemble_prediction)
print(final_FFV)

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective_light(trial, x_f, y_f), n_trials = 50)

In [ ]:
# completed_trials_ffv_light = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_ffv_light = sorted(completed_trials_ffv_light, key =lambda t: t.value)

In [ ]:
# sorted_ffv_light = [FrozenTrial(number=17, state=1, values=[0.005889850769456194], datetime_start=datetime.datetime(2025, 8, 5, 13, 45, 28, 523265), datetime_complete=datetime.datetime(2025, 8, 5, 13, 45, 52, 142508), params={'n_estimators': 393, 'learning_rate': 0.08332308279442384, 'num_leaves': 86, 'feature_fraction': 0.7157583462598314, 'bagging_fraction': 0.7954152811689044, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=17, value=None),
#  FrozenTrial(number=22, state=1, values=[0.005896544260028434], datetime_start=datetime.datetime(2025, 8, 5, 13, 47, 8, 497054), datetime_complete=datetime.datetime(2025, 8, 5, 13, 47, 32, 447539), params={'n_estimators': 425, 'learning_rate': 0.06253996819045345, 'num_leaves': 79, 'feature_fraction': 0.7549437125462386, 'bagging_fraction': 0.9170180182172016, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=22, value=None),
#  FrozenTrial(number=23, state=1, values=[0.00593053520530901], datetime_start=datetime.datetime(2025, 8, 5, 13, 47, 32, 449295), datetime_complete=datetime.datetime(2025, 8, 5, 13, 47, 53, 48661), params={'n_estimators': 433, 'learning_rate': 0.058526609289371964, 'num_leaves': 77, 'feature_fraction': 0.7642385970986028, 'bagging_fraction': 0.902175122923255, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=23, value=None),
#  FrozenTrial(number=42, state=1, values=[0.0059323178938870845], datetime_start=datetime.datetime(2025, 8, 5, 13, 54, 8, 350194), datetime_complete=datetime.datetime(2025, 8, 5, 13, 54, 24, 367848), params={'n_estimators': 387, 'learning_rate': 0.09447671572937201, 'num_leaves': 86, 'feature_fraction': 0.46465087852719633, 'bagging_fraction': 0.8879785494809, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=42, value=None),
#  FrozenTrial(number=41, state=1, values=[0.005935095999088026], datetime_start=datetime.datetime(2025, 8, 5, 13, 53, 52, 705760), datetime_complete=datetime.datetime(2025, 8, 5, 13, 54, 8, 348940), params={'n_estimators': 389, 'learning_rate': 0.08560506138209667, 'num_leaves': 83, 'feature_fraction': 0.49400643569606295, 'bagging_fraction': 0.8780944081424953, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=41, value=None),
#  FrozenTrial(number=32, state=1, values=[0.005937004253661891], datetime_start=datetime.datetime(2025, 8, 5, 13, 50, 56, 670462), datetime_complete=datetime.datetime(2025, 8, 5, 13, 51, 18, 626856), params={'n_estimators': 419, 'learning_rate': 0.0731915549261836, 'num_leaves': 81, 'feature_fraction': 0.6933105344558295, 'bagging_fraction': 0.8916671716476499, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=32, value=None),
#  FrozenTrial(number=15, state=1, values=[0.005945795159693399], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 43, 288765), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 58, 109650), params={'n_estimators': 394, 'learning_rate': 0.08159755562618098, 'num_leaves': 82, 'feature_fraction': 0.40840720809951603, 'bagging_fraction': 0.8716577337376731, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=15, value=None),
#  FrozenTrial(number=25, state=1, values=[0.005947749593635033], datetime_start=datetime.datetime(2025, 8, 5, 13, 48, 22, 94254), datetime_complete=datetime.datetime(2025, 8, 5, 13, 48, 50, 666061), params={'n_estimators': 434, 'learning_rate': 0.05864875571380497, 'num_leaves': 97, 'feature_fraction': 0.7708602523806604, 'bagging_fraction': 0.7469124549330084, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=25, value=None),
#  FrozenTrial(number=13, state=1, values=[0.00595544127833209], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 22, 1964), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 31, 706893), params={'n_estimators': 405, 'learning_rate': 0.07955823055291678, 'num_leaves': 52, 'feature_fraction': 0.4202517626933523, 'bagging_fraction': 0.9589383180619723, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=13, value=None),
#  FrozenTrial(number=38, state=1, values=[0.005955651144215681], datetime_start=datetime.datetime(2025, 8, 5, 13, 52, 58, 394417), datetime_complete=datetime.datetime(2025, 8, 5, 13, 53, 19, 625171), params={'n_estimators': 418, 'learning_rate': 0.06064968966142956, 'num_leaves': 101, 'feature_fraction': 0.6059625832637363, 'bagging_fraction': 0.9349666969701416, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=38, value=None),
#  FrozenTrial(number=36, state=1, values=[0.005963491134664246], datetime_start=datetime.datetime(2025, 8, 5, 13, 52, 22, 828865), datetime_complete=datetime.datetime(2025, 8, 5, 13, 52, 49, 624031), params={'n_estimators': 473, 'learning_rate': 0.07414397843999013, 'num_leaves': 89, 'feature_fraction': 0.7525710662833632, 'bagging_fraction': 0.817100805509821, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=36, value=None),
#  FrozenTrial(number=47, state=1, values=[0.005966178942290776], datetime_start=datetime.datetime(2025, 8, 5, 13, 55, 15, 561076), datetime_complete=datetime.datetime(2025, 8, 5, 13, 55, 34, 878896), params={'n_estimators': 444, 'learning_rate': 0.0950515431545593, 'num_leaves': 61, 'feature_fraction': 0.782599305179744, 'bagging_fraction': 0.8569903905052862, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=47, value=None),
#  FrozenTrial(number=16, state=1, values=[0.005981997449145215], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 58, 113017), datetime_complete=datetime.datetime(2025, 8, 5, 13, 45, 28, 521965), params={'n_estimators': 499, 'learning_rate': 0.06088368315048623, 'num_leaves': 121, 'feature_fraction': 0.6304738168266649, 'bagging_fraction': 0.9120053164693324, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=16, value=None),
#  FrozenTrial(number=21, state=1, values=[0.005994179313186257], datetime_start=datetime.datetime(2025, 8, 5, 13, 46, 56, 155097), datetime_complete=datetime.datetime(2025, 8, 5, 13, 47, 8, 496071), params={'n_estimators': 403, 'learning_rate': 0.08444685411986802, 'num_leaves': 55, 'feature_fraction': 0.5961298493001471, 'bagging_fraction': 0.9172716956691536, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=21, value=None),
#  FrozenTrial(number=28, state=1, values=[0.006000362151997614], datetime_start=datetime.datetime(2025, 8, 5, 13, 49, 37, 234268), datetime_complete=datetime.datetime(2025, 8, 5, 13, 50, 0, 524867), params={'n_estimators': 429, 'learning_rate': 0.05411257372777987, 'num_leaves': 92, 'feature_fraction': 0.7502291925210938, 'bagging_fraction': 0.8309701730029708, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=28, value=None),
#  FrozenTrial(number=26, state=1, values=[0.006000415713996578], datetime_start=datetime.datetime(2025, 8, 5, 13, 48, 50, 668008), datetime_complete=datetime.datetime(2025, 8, 5, 13, 49, 11, 732575), params={'n_estimators': 485, 'learning_rate': 0.0643576086811472, 'num_leaves': 73, 'feature_fraction': 0.7006582506696019, 'bagging_fraction': 0.800821403006202, 'min_child_samples': 20}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=26, value=None),
#  FrozenTrial(number=34, state=1, values=[0.0060073335197102186], datetime_start=datetime.datetime(2025, 8, 5, 13, 51, 35, 111159), datetime_complete=datetime.datetime(2025, 8, 5, 13, 51, 55, 417227), params={'n_estimators': 447, 'learning_rate': 0.06545796425575529, 'num_leaves': 93, 'feature_fraction': 0.5970088468543793, 'bagging_fraction': 0.8845738396131141, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=34, value=None),
#  FrozenTrial(number=35, state=1, values=[0.006008346320143382], datetime_start=datetime.datetime(2025, 8, 5, 13, 51, 55, 419435), datetime_complete=datetime.datetime(2025, 8, 5, 13, 52, 22, 827361), params={'n_estimators': 445, 'learning_rate': 0.06676208347008829, 'num_leaves': 97, 'feature_fraction': 0.8173751516903934, 'bagging_fraction': 0.7492063303127365, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=35, value=None),
#  FrozenTrial(number=48, state=1, values=[0.006010192497136933], datetime_start=datetime.datetime(2025, 8, 5, 13, 55, 34, 879965), datetime_complete=datetime.datetime(2025, 8, 5, 13, 55, 50, 374728), params={'n_estimators': 393, 'learning_rate': 0.07783366095586128, 'num_leaves': 100, 'feature_fraction': 0.4340618825453686, 'bagging_fraction': 0.7767799503019427, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=48, value=None),
#  FrozenTrial(number=14, state=1, values=[0.006020542382005003], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 31, 707854), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 43, 287328), params={'n_estimators': 400, 'learning_rate': 0.080765567843571, 'num_leaves': 73, 'feature_fraction': 0.40638706512716305, 'bagging_fraction': 0.8687836328114769, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=14, value=None),
#  FrozenTrial(number=33, state=1, values=[0.006021471362564619], datetime_start=datetime.datetime(2025, 8, 5, 13, 51, 18, 627730), datetime_complete=datetime.datetime(2025, 8, 5, 13, 51, 35, 109814), params={'n_estimators': 420, 'learning_rate': 0.07419290798767972, 'num_leaves': 67, 'feature_fraction': 0.6973055984968523, 'bagging_fraction': 0.9317549208208326, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=33, value=None),
#  FrozenTrial(number=29, state=1, values=[0.00602784456603914], datetime_start=datetime.datetime(2025, 8, 5, 13, 50, 0, 527225), datetime_complete=datetime.datetime(2025, 8, 5, 13, 50, 18, 917088), params={'n_estimators': 375, 'learning_rate': 0.06962525514357362, 'num_leaves': 78, 'feature_fraction': 0.6641053765373052, 'bagging_fraction': 0.9894311919978384, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=29, value=None),
#  FrozenTrial(number=20, state=1, values=[0.006031952312085622], datetime_start=datetime.datetime(2025, 8, 5, 13, 46, 33, 46257), datetime_complete=datetime.datetime(2025, 8, 5, 13, 46, 56, 153874), params={'n_estimators': 358, 'learning_rate': 0.08355092392591729, 'num_leaves': 83, 'feature_fraction': 0.9197646129313441, 'bagging_fraction': 0.8464197863145394, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=20, value=None),
#  FrozenTrial(number=31, state=1, values=[0.006031996720432994], datetime_start=datetime.datetime(2025, 8, 5, 13, 50, 32, 751099), datetime_complete=datetime.datetime(2025, 8, 5, 13, 50, 56, 668891), params={'n_estimators': 387, 'learning_rate': 0.08788040231359408, 'num_leaves': 80, 'feature_fraction': 0.9069043386722527, 'bagging_fraction': 0.8747874489105776, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=31, value=None),
#  FrozenTrial(number=46, state=1, values=[0.006045805470417717], datetime_start=datetime.datetime(2025, 8, 5, 13, 55, 5, 330378), datetime_complete=datetime.datetime(2025, 8, 5, 13, 55, 15, 559907), params={'n_estimators': 344, 'learning_rate': 0.09333353995154649, 'num_leaves': 70, 'feature_fraction': 0.4798885440929813, 'bagging_fraction': 0.9581806980636581, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=46, value=None),
#  FrozenTrial(number=19, state=1, values=[0.006061358462851042], datetime_start=datetime.datetime(2025, 8, 5, 13, 46, 8, 806010), datetime_complete=datetime.datetime(2025, 8, 5, 13, 46, 33, 44827), params={'n_estimators': 410, 'learning_rate': 0.09897635150956494, 'num_leaves': 107, 'feature_fraction': 0.7113132772865168, 'bagging_fraction': 0.7730606161745054, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=19, value=None),
#  FrozenTrial(number=0, state=1, values=[0.0060662737637195195], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 4, 593121), datetime_complete=datetime.datetime(2025, 8, 5, 13, 42, 16, 137130), params={'n_estimators': 327, 'learning_rate': 0.0912767466302197, 'num_leaves': 63, 'feature_fraction': 0.5098616259507204, 'bagging_fraction': 0.6891950990417905, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None),
#  FrozenTrial(number=24, state=1, values=[0.006067704626438917], datetime_start=datetime.datetime(2025, 8, 5, 13, 47, 53, 49881), datetime_complete=datetime.datetime(2025, 8, 5, 13, 48, 22, 91316), params={'n_estimators': 430, 'learning_rate': 0.0386132903977162, 'num_leaves': 105, 'feature_fraction': 0.789513100489857, 'bagging_fraction': 0.9110078715395747, 'min_child_samples': 15}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=24, value=None),
#  FrozenTrial(number=43, state=1, values=[0.006076290270756086], datetime_start=datetime.datetime(2025, 8, 5, 13, 54, 24, 369025), datetime_complete=datetime.datetime(2025, 8, 5, 13, 54, 39, 940966), params={'n_estimators': 384, 'learning_rate': 0.09971292903421003, 'num_leaves': 86, 'feature_fraction': 0.4976651269656365, 'bagging_fraction': 0.8449835370409096, 'min_child_samples': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=43, value=None),
#  FrozenTrial(number=39, state=1, values=[0.006082734139717908], datetime_start=datetime.datetime(2025, 8, 5, 13, 53, 19, 627582), datetime_complete=datetime.datetime(2025, 8, 5, 13, 53, 36, 417569), params={'n_estimators': 462, 'learning_rate': 0.08951184942414524, 'num_leaves': 67, 'feature_fraction': 0.73128384198105, 'bagging_fraction': 0.9998870916868154, 'min_child_samples': 32}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=39, value=None),
#  FrozenTrial(number=30, state=1, values=[0.006089706278604451], datetime_start=datetime.datetime(2025, 8, 5, 13, 50, 18, 918507), datetime_complete=datetime.datetime(2025, 8, 5, 13, 50, 32, 750096), params={'n_estimators': 322, 'learning_rate': 0.056815058667255815, 'num_leaves': 67, 'feature_fraction': 0.807766077378054, 'bagging_fraction': 0.6827884739242898, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=30, value=None),
#  FrozenTrial(number=40, state=1, values=[0.006091187332734078], datetime_start=datetime.datetime(2025, 8, 5, 13, 53, 36, 420054), datetime_complete=datetime.datetime(2025, 8, 5, 13, 53, 52, 704581), params={'n_estimators': 338, 'learning_rate': 0.07740935606898557, 'num_leaves': 55, 'feature_fraction': 0.8874481463785848, 'bagging_fraction': 0.7018396525738521, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=40, value=None),
#  FrozenTrial(number=44, state=1, values=[0.006147314280635584], datetime_start=datetime.datetime(2025, 8, 5, 13, 54, 39, 943924), datetime_complete=datetime.datetime(2025, 8, 5, 13, 54, 51, 108971), params={'n_estimators': 352, 'learning_rate': 0.09312891200152441, 'num_leaves': 75, 'feature_fraction': 0.4668867432555316, 'bagging_fraction': 0.9499738660881025, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=44, value=None),
#  FrozenTrial(number=6, state=1, values=[0.006150763250825675], datetime_start=datetime.datetime(2025, 8, 5, 13, 43, 7, 641845), datetime_complete=datetime.datetime(2025, 8, 5, 13, 43, 17, 515467), params={'n_estimators': 457, 'learning_rate': 0.0714157994763478, 'num_leaves': 42, 'feature_fraction': 0.4669049154043016, 'bagging_fraction': 0.9671602526878349, 'min_child_samples': 36}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=6, value=None),
#  FrozenTrial(number=18, state=1, values=[0.0061678657133062255], datetime_start=datetime.datetime(2025, 8, 5, 13, 45, 52, 143741), datetime_complete=datetime.datetime(2025, 8, 5, 13, 46, 8, 804531), params={'n_estimators': 371, 'learning_rate': 0.04465591129759965, 'num_leaves': 86, 'feature_fraction': 0.7327442183793089, 'bagging_fraction': 0.7853239609696975, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=18, value=None),
#  FrozenTrial(number=7, state=1, values=[0.006169783739748786], datetime_start=datetime.datetime(2025, 8, 5, 13, 43, 17, 516548), datetime_complete=datetime.datetime(2025, 8, 5, 13, 43, 24, 625035), params={'n_estimators': 391, 'learning_rate': 0.09598252930413577, 'num_leaves': 34, 'feature_fraction': 0.5392186928187025, 'bagging_fraction': 0.7302691237205639, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=7, value=None),
#  FrozenTrial(number=27, state=1, values=[0.006177135096272288], datetime_start=datetime.datetime(2025, 8, 5, 13, 49, 11, 733763), datetime_complete=datetime.datetime(2025, 8, 5, 13, 49, 37, 231793), params={'n_estimators': 371, 'learning_rate': 0.035108834709830825, 'num_leaves': 111, 'feature_fraction': 0.8509116865492008, 'bagging_fraction': 0.9246474784227302, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=27, value=None),
#  FrozenTrial(number=8, state=1, values=[0.006233481585375524], datetime_start=datetime.datetime(2025, 8, 5, 13, 43, 24, 626811), datetime_complete=datetime.datetime(2025, 8, 5, 13, 43, 50, 688706), params={'n_estimators': 475, 'learning_rate': 0.025401798003087547, 'num_leaves': 60, 'feature_fraction': 0.9718449189455374, 'bagging_fraction': 0.9778786750943567, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=8, value=None),
#  FrozenTrial(number=45, state=1, values=[0.00631693129321128], datetime_start=datetime.datetime(2025, 8, 5, 13, 54, 51, 110140), datetime_complete=datetime.datetime(2025, 8, 5, 13, 55, 5, 329200), params={'n_estimators': 379, 'learning_rate': 0.08654470844639467, 'num_leaves': 89, 'feature_fraction': 0.5663510772602502, 'bagging_fraction': 0.897408355971832, 'min_child_samples': 42}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=45, value=None),
#  FrozenTrial(number=11, state=1, values=[0.006323874109726992], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 8, 272428), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 11, 907064), params={'n_estimators': 361, 'learning_rate': 0.07165489450645687, 'num_leaves': 20, 'feature_fraction': 0.41477210403376014, 'bagging_fraction': 0.8468850443489486, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=11, value=None),
#  FrozenTrial(number=49, state=1, values=[0.006328814892982135], datetime_start=datetime.datetime(2025, 8, 5, 13, 55, 50, 375975), datetime_complete=datetime.datetime(2025, 8, 5, 13, 56, 2, 268421), params={'n_estimators': 288, 'learning_rate': 0.04593163928106782, 'num_leaves': 112, 'feature_fraction': 0.4511412559694472, 'bagging_fraction': 0.46614749323278803, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=49, value=None),
#  FrozenTrial(number=12, state=1, values=[0.006342712989989748], datetime_start=datetime.datetime(2025, 8, 5, 13, 44, 11, 908227), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 22, 713), params={'n_estimators': 309, 'learning_rate': 0.05021479518414476, 'num_leaves': 61, 'feature_fraction': 0.5385127506630338, 'bagging_fraction': 0.8088451494923004, 'min_child_samples': 36}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=12, value=None),
#  FrozenTrial(number=1, state=1, values=[0.006349031576785972], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 16, 138589), datetime_complete=datetime.datetime(2025, 8, 5, 13, 42, 24, 792225), params={'n_estimators': 436, 'learning_rate': 0.0770112365551712, 'num_leaves': 40, 'feature_fraction': 0.6442767395980267, 'bagging_fraction': 0.6804515231465469, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=1, value=None),
#  FrozenTrial(number=2, state=1, values=[0.006388008761483075], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 24, 793550), datetime_complete=datetime.datetime(2025, 8, 5, 13, 42, 36, 941032), params={'n_estimators': 454, 'learning_rate': 0.09202867703626615, 'num_leaves': 95, 'feature_fraction': 0.7810604901131596, 'bagging_fraction': 0.5455676838796064, 'min_child_samples': 44}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=2, value=None),
#  FrozenTrial(number=10, state=1, values=[0.0063930152172951785], datetime_start=datetime.datetime(2025, 8, 5, 13, 43, 57, 28673), datetime_complete=datetime.datetime(2025, 8, 5, 13, 44, 8, 271228), params={'n_estimators': 337, 'learning_rate': 0.04729053826362638, 'num_leaves': 71, 'feature_fraction': 0.6456278419685164, 'bagging_fraction': 0.4018843037710959, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=10, value=None),
#  FrozenTrial(number=37, state=1, values=[0.006394742197259669], datetime_start=datetime.datetime(2025, 8, 5, 13, 52, 49, 625162), datetime_complete=datetime.datetime(2025, 8, 5, 13, 52, 58, 393227), params={'n_estimators': 225, 'learning_rate': 0.05264276219606725, 'num_leaves': 78, 'feature_fraction': 0.6702002882851292, 'bagging_fraction': 0.6403264570422557, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=37, value=None),
#  FrozenTrial(number=4, state=1, values=[0.006429178906476622], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 43, 848986), datetime_complete=datetime.datetime(2025, 8, 5, 13, 42, 48, 799170), params={'n_estimators': 260, 'learning_rate': 0.09149622119831229, 'num_leaves': 46, 'feature_fraction': 0.4677833094090613, 'bagging_fraction': 0.6477071909359853, 'min_child_samples': 29}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=4, value=None),
#  FrozenTrial(number=5, state=1, values=[0.006460349360231785], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 48, 800803), datetime_complete=datetime.datetime(2025, 8, 5, 13, 43, 7, 640649), params={'n_estimators': 456, 'learning_rate': 0.022361762209623534, 'num_leaves': 88, 'feature_fraction': 0.8587481471198926, 'bagging_fraction': 0.582987483140303, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=5, value=None),
#  FrozenTrial(number=3, state=1, values=[0.006706015507614232], datetime_start=datetime.datetime(2025, 8, 5, 13, 42, 36, 942498), datetime_complete=datetime.datetime(2025, 8, 5, 13, 42, 43, 848000), params={'n_estimators': 222, 'learning_rate': 0.06543342194418109, 'num_leaves': 102, 'feature_fraction': 0.8367730152408753, 'bagging_fraction': 0.5047734557924556, 'min_child_samples': 50}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=3, value=None),
#  FrozenTrial(number=9, state=1, values=[0.007724936291709122], datetime_start=datetime.datetime(2025, 8, 5, 13, 43, 50, 689887), datetime_complete=datetime.datetime(2025, 8, 5, 13, 43, 57, 27705), params={'n_estimators': 291, 'learning_rate': 0.01326380722084659, 'num_leaves': 121, 'feature_fraction': 0.5382180945873916, 'bagging_fraction': 0.5498074440571501, 'min_child_samples': 43}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=9, value=None)]

In [ ]:
# desc_ffv = desc_test[common_cols3]
# top_trials = sorted_ffv_light[:15]
# all_predictions = []
# FFV_light = []
# for i, trial in enumerate(top_trials):
#     params = trial.params
#     params['objective'] = 'mae'
#     params['metric'] = 'mae'
#     params['bagging_freq'] = 1
#     params['n_jobs'] = -1
#     params['verbosity'] = -1
#     params['random_state'] = 42
#     model = Model_light(params)
#     model.train(x_f, y_f)
#     preds = model.evalution()
#     all_predictions.append(preds)
#     FFV_light.append(model.prediction(desc_ffv))
#     print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
# final_ensemble_prediction = np.mean(all_predictions, axis=0)
# final_FFV_light = np.mean(FFV_light, axis=0)
# print(final_ensemble_prediction)
# print(final_FFV_light)

**Tc**

In [ ]:
train_cols4 = set(tc.columns) - {"Tc"}
test_cols4 = set(desc_test.columns)
common_cols4 = list(train_cols4 & test_cols4)
x_tc=tc[common_cols4].copy()
y_tc=tc["Tc"].copy()

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective(trial, x_tc, y_tc), n_trials = 50)

In [ ]:
# completed_trials_tc = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_tc = sorted(completed_trials_tc, key =lambda t: t.value)

In [ ]:
sorted_tc = [FrozenTrial(number=47, state=1, values=[0.03081004089599245], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 49, 25997), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 54, 469399), params={'n_estimators': 423, 'learning_rate': 0.09481006008981932, 'max_depth': 4, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=47, value=None),
 FrozenTrial(number=39, state=1, values=[0.03158562385279141], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 5, 636755), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 11, 233759), params={'n_estimators': 455, 'learning_rate': 0.06451926546340847, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=39, value=None),
 FrozenTrial(number=44, state=1, values=[0.03174065685631631], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 36, 560304), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 45, 172769), params={'n_estimators': 428, 'learning_rate': 0.09746608117285409, 'max_depth': 5, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=44, value=None),
 FrozenTrial(number=41, state=1, values=[0.032397120082711846], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 13, 290905), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 21, 142998), params={'n_estimators': 452, 'learning_rate': 0.06415363170784565, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=41, value=None),
 FrozenTrial(number=43, state=1, values=[0.032461041884286126], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 28, 901792), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 36, 558112), params={'n_estimators': 451, 'learning_rate': 0.09782695756608778, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=43, value=None),
 FrozenTrial(number=42, state=1, values=[0.03261765732109993], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 21, 145453), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 28, 900849), params={'n_estimators': 452, 'learning_rate': 0.08236724450078674, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=42, value=None),
 FrozenTrial(number=36, state=1, values=[0.0339078497334366], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 57, 460031), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 1, 301273), params={'n_estimators': 435, 'learning_rate': 0.057391296124677235, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=36, value=None),
 FrozenTrial(number=9, state=1, values=[0.034317003748077585], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 23, 327133), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 33, 698972), params={'n_estimators': 457, 'learning_rate': 0.018124507083627124, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=9, value=None),
 FrozenTrial(number=34, state=1, values=[0.03473484651461969], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 50, 31054), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 54, 944074), params={'n_estimators': 461, 'learning_rate': 0.04368805978185314, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=34, value=None),
 FrozenTrial(number=24, state=1, values=[0.03516792304654246], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 13, 538144), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 17, 241577), params={'n_estimators': 275, 'learning_rate': 0.04053387647093807, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=18, state=1, values=[0.03542078013022741], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 52, 650937), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 57, 967604), params={'n_estimators': 378, 'learning_rate': 0.032377261841257096, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=18, value=None),
 FrozenTrial(number=21, state=1, values=[0.03553516688498665], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 1, 654393), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 7, 718920), params={'n_estimators': 314, 'learning_rate': 0.038685461258396926, 'max_depth': 9, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=21, value=None),
 FrozenTrial(number=11, state=1, values=[0.03555157083956044], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 37, 598254), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 40, 983229), params={'n_estimators': 240, 'learning_rate': 0.053089332357190454, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=11, value=None),
 FrozenTrial(number=31, state=1, values=[0.035574235565117684], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 32, 853303), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 38, 952174), params={'n_estimators': 319, 'learning_rate': 0.0404753025930024, 'max_depth': 9, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=31, value=None),
 FrozenTrial(number=32, state=1, values=[0.03572143358184157], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 38, 955818), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 43, 764464), params={'n_estimators': 292, 'learning_rate': 0.03521944588951363, 'max_depth': 10, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=32, value=None),
 FrozenTrial(number=10, state=1, values=[0.035722188602136766], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 33, 701448), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 37, 596994), params={'n_estimators': 251, 'learning_rate': 0.044674919508313246, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=10, value=None),
 FrozenTrial(number=25, state=1, values=[0.036734063455272706], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 17, 245008), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 21, 712492), params={'n_estimators': 278, 'learning_rate': 0.024122132430005452, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=25, value=None),
 FrozenTrial(number=2, state=1, values=[0.0369614916409446], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 9, 128603), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 12, 262639), params={'n_estimators': 446, 'learning_rate': 0.09033273346409298, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=2, value=None),
 FrozenTrial(number=33, state=1, values=[0.03711726707412063], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 43, 767225), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 50, 28289), params={'n_estimators': 262, 'learning_rate': 0.020616566413388833, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=33, value=None),
 FrozenTrial(number=15, state=1, values=[0.037170562800977115], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 47, 10725), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 49, 56436), params={'n_estimators': 200, 'learning_rate': 0.0754255361875784, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=15, value=None),
 FrozenTrial(number=13, state=1, values=[0.03845047560729338], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 42, 187869), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 44, 817110), params={'n_estimators': 290, 'learning_rate': 0.07130145099642757, 'max_depth': 8, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 7, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=13, value=None),
 FrozenTrial(number=22, state=1, values=[0.038738902117563104], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 7, 721644), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 11, 640003), params={'n_estimators': 310, 'learning_rate': 0.03791116584155252, 'max_depth': 9, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 8, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=22, value=None),
 FrozenTrial(number=28, state=1, values=[0.04004706711273515], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 25, 539692), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 30, 431566), params={'n_estimators': 359, 'learning_rate': 0.01816117243297467, 'max_depth': 7, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=28, value=None),
 FrozenTrial(number=49, state=1, values=[0.053355252916580734], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 56, 383231), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 58, 684905), params={'n_estimators': 481, 'learning_rate': 0.09512500941133455, 'max_depth': 4, 'min_child_weight': 8, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=49, value=None),
 FrozenTrial(number=29, state=1, values=[0.053925411476312066], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 30, 434673), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 31, 746678), params={'n_estimators': 270, 'learning_rate': 0.05964062556925248, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 6, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=29, value=None),
 FrozenTrial(number=48, state=1, values=[0.05558965338852522], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 54, 471795), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 56, 380358), params={'n_estimators': 423, 'learning_rate': 0.09921896073356966, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=48, value=None),
 FrozenTrial(number=35, state=1, values=[0.05603803133799343], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 54, 946340), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 57, 457381), params={'n_estimators': 467, 'learning_rate': 0.02843069538440727, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=35, value=None),
 FrozenTrial(number=0, state=1, values=[0.05626191040206045], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 4, 803887), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 7, 128047), params={'n_estimators': 455, 'learning_rate': 0.0651812101610992, 'max_depth': 8, 'min_child_weight': 10, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=30, state=1, values=[0.058367390645264694], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 31, 748571), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 32, 851017), params={'n_estimators': 210, 'learning_rate': 0.04864556624849285, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=30, value=None),
 FrozenTrial(number=16, state=1, values=[0.05844498877703921], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 49, 59102), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 50, 508841), params={'n_estimators': 306, 'learning_rate': 0.03324082876063593, 'max_depth': 9, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 8, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=19, state=1, values=[0.05852421414910185], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 57, 968939), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 59, 789025), params={'n_estimators': 384, 'learning_rate': 0.029135748453459582, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 1, 'reg_lambda': 5, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=19, value=None),
 FrozenTrial(number=23, state=1, values=[0.05872350554376953], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 11, 642429), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 13, 535657), params={'n_estimators': 366, 'learning_rate': 0.026172048369485175, 'max_depth': 9, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=23, value=None),
 FrozenTrial(number=12, state=1, values=[0.06004096599902107], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 40, 984449), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 42, 184902), params={'n_estimators': 236, 'learning_rate': 0.06099463869531931, 'max_depth': 10, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=12, value=None),
 FrozenTrial(number=37, state=1, values=[0.060125866116343366], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 1, 303965), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 3, 560513), params={'n_estimators': 435, 'learning_rate': 0.06835439050917698, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 5, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=37, value=None),
 FrozenTrial(number=4, state=1, values=[0.06105842557105232], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 14, 420432), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 16, 167180), params={'n_estimators': 381, 'learning_rate': 0.03420334201050986, 'max_depth': 9, 'min_child_weight': 8, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=4, value=None),
 FrozenTrial(number=26, state=1, values=[0.06110932760283295], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 21, 713736), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 23, 792481), params={'n_estimators': 474, 'learning_rate': 0.04061568190898391, 'max_depth': 8, 'min_child_weight': 5, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=26, value=None),
 FrozenTrial(number=40, state=1, values=[0.06889919061665233], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 11, 236558), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 13, 288615), params={'n_estimators': 480, 'learning_rate': 0.07737666057771486, 'max_depth': 4, 'min_child_weight': 6, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=27, state=1, values=[0.06997310378033571], datetime_start=datetime.datetime(2025, 8, 5, 13, 58, 23, 795328), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 25, 537009), params={'n_estimators': 330, 'learning_rate': 0.03194578298781574, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 2, 'reg_lambda': 5, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=27, value=None),
 FrozenTrial(number=1, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 7, 129187), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 9, 125607), params={'n_estimators': 463, 'learning_rate': 0.019024420952307948, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 5, 'reg_lambda': 3, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=1, value=None),
 FrozenTrial(number=3, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 12, 264991), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 14, 417895), params={'n_estimators': 401, 'learning_rate': 0.023291247597117953, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 3, 'reg_lambda': 1, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=3, value=None),
 FrozenTrial(number=5, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 16, 168605), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 18, 278553), params={'n_estimators': 440, 'learning_rate': 0.012870729178214947, 'max_depth': 6, 'min_child_weight': 9, 'gamma': 2, 'reg_lambda': 8, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=5, value=None),
 FrozenTrial(number=6, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 18, 280040), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 20, 122242), params={'n_estimators': 425, 'learning_rate': 0.0916463721355373, 'max_depth': 4, 'min_child_weight': 10, 'gamma': 3, 'reg_lambda': 10, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=6, value=None),
 FrozenTrial(number=7, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 20, 123696), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 21, 725379), params={'n_estimators': 346, 'learning_rate': 0.018103587405495402, 'max_depth': 7, 'min_child_weight': 9, 'gamma': 4, 'reg_lambda': 5, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=7, value=None),
 FrozenTrial(number=8, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 21, 726648), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 23, 325743), params={'n_estimators': 339, 'learning_rate': 0.08928829060167459, 'max_depth': 3, 'min_child_weight': 6, 'gamma': 5, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=14, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 44, 820485), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 47, 7932), params={'n_estimators': 496, 'learning_rate': 0.045303694429942705, 'max_depth': 10, 'min_child_weight': 5, 'gamma': 2, 'reg_lambda': 10, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=14, value=None),
 FrozenTrial(number=17, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 50, 511236), datetime_complete=datetime.datetime(2025, 8, 5, 13, 57, 52, 649929), params={'n_estimators': 499, 'learning_rate': 0.05073025546561882, 'max_depth': 7, 'min_child_weight': 5, 'gamma': 2, 'reg_lambda': 9, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=17, value=None),
 FrozenTrial(number=20, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 57, 59, 789880), datetime_complete=datetime.datetime(2025, 8, 5, 13, 58, 1, 652899), params={'n_estimators': 410, 'learning_rate': 0.0105185421136397, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 4, 'reg_lambda': 6, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=20, value=None),
 FrozenTrial(number=38, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 3, 562931), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 5, 634421), params={'n_estimators': 407, 'learning_rate': 0.05757792424724691, 'max_depth': 6, 'min_child_weight': 7, 'gamma': 3, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=38, value=None),
 FrozenTrial(number=45, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 45, 175171), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 47, 182947), params={'n_estimators': 450, 'learning_rate': 0.09905556204754376, 'max_depth': 5, 'min_child_weight': 9, 'gamma': 4, 'reg_lambda': 2, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=45, value=None),
 FrozenTrial(number=46, state=1, values=[0.07520017763327125], datetime_start=datetime.datetime(2025, 8, 5, 13, 59, 47, 184326), datetime_complete=datetime.datetime(2025, 8, 5, 13, 59, 49, 23557), params={'n_estimators': 415, 'learning_rate': 0.08563864147664395, 'max_depth': 4, 'min_child_weight': 8, 'gamma': 5, 'reg_lambda': 2, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=46, value=None)]

In [ ]:
desc_tc = desc_test[common_cols4]
top_trials = sorted_tc[:10]
all_predictions = []
TC = []
for i, trial in enumerate(top_trials):
    params = trial.params
    params['random_state'] = 42+i
    model = Model_XG(params)
    model.train(x_tc, y_tc)
    preds = model.evalution()
    all_predictions.append(preds)
    TC.append(model.prediction(desc_tc))
    print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
final_ensemble_prediction = np.mean(all_predictions, axis=0)
final_TC = np.mean(TC, axis=0)
print(final_ensemble_prediction)
print(final_TC)

In [ ]:
 # study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective_light(trial, x_tc, y_tc), n_trials = 50)

In [ ]:
# completed_trials_tc_light = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_tc_light = sorted(completed_trials_tc_light, key =lambda t: t.value)

In [ ]:
# sorted_tc_light = [FrozenTrial(number=36, state=1, values=[0.027855714090278674], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 41, 597098), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 46, 334704), params={'n_estimators': 340, 'learning_rate': 0.06404986363520757, 'num_leaves': 97, 'feature_fraction': 0.6484228816516407, 'bagging_fraction': 0.9252931832493545, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=36, value=None),
#  FrozenTrial(number=34, state=1, values=[0.02792582594320053], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 32, 459455), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 38, 285958), params={'n_estimators': 372, 'learning_rate': 0.07362697898159776, 'num_leaves': 98, 'feature_fraction': 0.6830783699076439, 'bagging_fraction': 0.9303567655510827, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=34, value=None),
#  FrozenTrial(number=37, state=1, values=[0.028039567089629295], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 46, 336228), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 52, 635892), params={'n_estimators': 440, 'learning_rate': 0.06542107533135284, 'num_leaves': 99, 'feature_fraction': 0.6525486179080678, 'bagging_fraction': 0.7387865826275977, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=37, value=None),
#  FrozenTrial(number=39, state=1, values=[0.028062382456813438], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 59, 222130), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 2, 889443), params={'n_estimators': 498, 'learning_rate': 0.07099251119962419, 'num_leaves': 85, 'feature_fraction': 0.7521943380761672, 'bagging_fraction': 0.7862256314400293, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=39, value=None),
#  FrozenTrial(number=0, state=1, values=[0.028111599954950016], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 43, 505041), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 47, 361561), params={'n_estimators': 402, 'learning_rate': 0.05729710062118903, 'num_leaves': 29, 'feature_fraction': 0.6401151541331851, 'bagging_fraction': 0.5598693138372158, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None),
#  FrozenTrial(number=23, state=1, values=[0.028169302862993476], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 44, 510028), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 48, 282477), params={'n_estimators': 355, 'learning_rate': 0.058147554457794154, 'num_leaves': 110, 'feature_fraction': 0.6319496480890556, 'bagging_fraction': 0.7148834930179402, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=23, value=None),
#  FrozenTrial(number=43, state=1, values=[0.028190863947063067], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 22, 560251), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 34, 804984), params={'n_estimators': 496, 'learning_rate': 0.06332172271718618, 'num_leaves': 74, 'feature_fraction': 0.8213104136004852, 'bagging_fraction': 0.8342882727138556, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=43, value=None),
#  FrozenTrial(number=32, state=1, values=[0.02825791984863991], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 20, 474351), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 25, 203061), params={'n_estimators': 344, 'learning_rate': 0.02464028131355306, 'num_leaves': 120, 'feature_fraction': 0.9936612268539541, 'bagging_fraction': 0.9357761289557764, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=32, value=None),
#  FrozenTrial(number=44, state=1, values=[0.028266403666257735], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 34, 806148), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 36, 192696), params={'n_estimators': 442, 'learning_rate': 0.07850746000244468, 'num_leaves': 79, 'feature_fraction': 0.5316293705799562, 'bagging_fraction': 0.7732754074967965, 'min_child_samples': 32}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=44, value=None),
#  FrozenTrial(number=20, state=1, values=[0.028377603808842137], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 28, 812929), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 35, 533236), params={'n_estimators': 355, 'learning_rate': 0.07870435777228063, 'num_leaves': 107, 'feature_fraction': 0.9093913051590091, 'bagging_fraction': 0.8904847867713573, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=20, value=None),
#  FrozenTrial(number=46, state=1, values=[0.028430703254846954], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 39, 393478), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 46, 89615), params={'n_estimators': 403, 'learning_rate': 0.0709152674178611, 'num_leaves': 99, 'feature_fraction': 0.6515727674682255, 'bagging_fraction': 0.7377097807026919, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=46, value=None),
#  FrozenTrial(number=38, state=1, values=[0.028434910861786324], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 52, 640019), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 59, 220715), params={'n_estimators': 456, 'learning_rate': 0.06631620288706841, 'num_leaves': 99, 'feature_fraction': 0.6623723938946273, 'bagging_fraction': 0.7424683371136441, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=38, value=None),
#  FrozenTrial(number=18, state=1, values=[0.028443623969076754], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 24, 71045), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 26, 474826), params={'n_estimators': 310, 'learning_rate': 0.021201946735181464, 'num_leaves': 123, 'feature_fraction': 0.898142671433282, 'bagging_fraction': 0.877955622300473, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=18, value=None),
#  FrozenTrial(number=1, state=1, values=[0.02849557090854098], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 47, 362833), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 48, 568002), params={'n_estimators': 209, 'learning_rate': 0.040123752565153735, 'num_leaves': 26, 'feature_fraction': 0.9593928820210593, 'bagging_fraction': 0.994281832651654, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=1, value=None),
#  FrozenTrial(number=22, state=1, values=[0.028499129278676375], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 40, 970026), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 44, 508817), params={'n_estimators': 304, 'learning_rate': 0.09032978425980566, 'num_leaves': 130, 'feature_fraction': 0.9427275111486599, 'bagging_fraction': 0.8874773103141527, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=22, value=None),
#  FrozenTrial(number=29, state=1, values=[0.02851694411770367], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 12, 67377), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 16, 766741), params={'n_estimators': 409, 'learning_rate': 0.043275841732416796, 'num_leaves': 112, 'feature_fraction': 0.7889071134292486, 'bagging_fraction': 0.9985408338858269, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=29, value=None),
#  FrozenTrial(number=13, state=1, values=[0.028535520711187815], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 17, 189474), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 17, 929629), params={'n_estimators': 202, 'learning_rate': 0.0679977804274105, 'num_leaves': 48, 'feature_fraction': 0.6148263388757668, 'bagging_fraction': 0.9546520644024217, 'min_child_samples': 47}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=13, value=None),
#  FrozenTrial(number=35, state=1, values=[0.02856304446236125], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 38, 288013), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 41, 595598), params={'n_estimators': 376, 'learning_rate': 0.05529229975510647, 'num_leaves': 94, 'feature_fraction': 0.6927756675394882, 'bagging_fraction': 0.8460751875279315, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=35, value=None),
#  FrozenTrial(number=40, state=1, values=[0.02859856921731762], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 2, 891033), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 6, 429075), params={'n_estimators': 485, 'learning_rate': 0.07143215720868065, 'num_leaves': 83, 'feature_fraction': 0.7445753710328039, 'bagging_fraction': 0.7769967828741106, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=40, value=None),
#  FrozenTrial(number=9, state=1, values=[0.028623788619012165], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 12, 319002), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 12, 969339), params={'n_estimators': 334, 'learning_rate': 0.052440575777010424, 'num_leaves': 65, 'feature_fraction': 0.5473669206584723, 'bagging_fraction': 0.4051314863518881, 'min_child_samples': 28}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=9, value=None),
#  FrozenTrial(number=19, state=1, values=[0.028710365114417633], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 26, 475999), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 28, 811677), params={'n_estimators': 315, 'learning_rate': 0.01994852088358099, 'num_leaves': 123, 'feature_fraction': 0.8450104279240143, 'bagging_fraction': 0.8875815327515115, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=19, value=None),
#  FrozenTrial(number=26, state=1, values=[0.028764643495546006], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 57, 65744), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 0, 823354), params={'n_estimators': 452, 'learning_rate': 0.06039460390621562, 'num_leaves': 104, 'feature_fraction': 0.6595925652405715, 'bagging_fraction': 0.6028843556818749, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=26, value=None),
#  FrozenTrial(number=14, state=1, values=[0.028806521182130663], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 17, 930819), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 18, 931557), params={'n_estimators': 285, 'learning_rate': 0.03936951371842094, 'num_leaves': 34, 'feature_fraction': 0.877952815327262, 'bagging_fraction': 0.6295588859943276, 'min_child_samples': 34}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=14, value=None),
#  FrozenTrial(number=31, state=1, values=[0.028807543167944094], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 18, 408715), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 20, 473438), params={'n_estimators': 305, 'learning_rate': 0.038439545593050434, 'num_leaves': 129, 'feature_fraction': 0.9069790952518862, 'bagging_fraction': 0.8517308561439889, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=31, value=None),
#  FrozenTrial(number=21, state=1, values=[0.028812889601809954], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 35, 534474), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 40, 968588), params={'n_estimators': 366, 'learning_rate': 0.07909331923595302, 'num_leaves': 111, 'feature_fraction': 0.8729901180821757, 'bagging_fraction': 0.8881760494526675, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=21, value=None),
#  FrozenTrial(number=25, state=1, values=[0.028821459062780865], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 54, 934767), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 57, 64552), params={'n_estimators': 355, 'learning_rate': 0.07410172492570087, 'num_leaves': 87, 'feature_fraction': 0.5597060381925432, 'bagging_fraction': 0.4361183150292, 'min_child_samples': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=25, value=None),
#  FrozenTrial(number=12, state=1, values=[0.028855960249569997], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 15, 532101), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 17, 183298), params={'n_estimators': 278, 'learning_rate': 0.028766375470731076, 'num_leaves': 39, 'feature_fraction': 0.8572292938708402, 'bagging_fraction': 0.6052459957300921, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=12, value=None),
#  FrozenTrial(number=45, state=1, values=[0.028918348583171136], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 36, 193713), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 39, 392341), params={'n_estimators': 475, 'learning_rate': 0.049919472884200504, 'num_leaves': 93, 'feature_fraction': 0.5935285698801495, 'bagging_fraction': 0.9747579318294163, 'min_child_samples': 20}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=45, value=None),
#  FrozenTrial(number=16, state=1, values=[0.028954753869123714], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 21, 958096), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 23, 469092), params={'n_estimators': 386, 'learning_rate': 0.017328358267308483, 'num_leaves': 34, 'feature_fraction': 0.5990806508171314, 'bagging_fraction': 0.6684209432010939, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=16, value=None),
#  FrozenTrial(number=8, state=1, values=[0.02895980346890258], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 59, 775293), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 12, 317614), params={'n_estimators': 489, 'learning_rate': 0.06495230113661601, 'num_leaves': 78, 'feature_fraction': 0.9511962435631603, 'bagging_fraction': 0.7394478069855297, 'min_child_samples': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=8, value=None),
#  FrozenTrial(number=6, state=1, values=[0.02902966608675377], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 53, 978377), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 58, 524554), params={'n_estimators': 462, 'learning_rate': 0.0836154331528506, 'num_leaves': 118, 'feature_fraction': 0.4039198741102275, 'bagging_fraction': 0.8098375865729475, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=6, value=None),
#  FrozenTrial(number=48, state=1, values=[0.02903120971282476], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 50, 143189), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 53, 701556), params={'n_estimators': 440, 'learning_rate': 0.07450398373723177, 'num_leaves': 87, 'feature_fraction': 0.45082571046337777, 'bagging_fraction': 0.805280301505972, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=48, value=None),
#  FrozenTrial(number=7, state=1, values=[0.029042127391456053], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 58, 525713), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 59, 774268), params={'n_estimators': 426, 'learning_rate': 0.06912510891214706, 'num_leaves': 77, 'feature_fraction': 0.4854336002302916, 'bagging_fraction': 0.8007230386812687, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=7, value=None),
#  FrozenTrial(number=2, state=1, values=[0.029050807675862798], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 48, 569376), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 49, 752548), params={'n_estimators': 343, 'learning_rate': 0.03320761423497493, 'num_leaves': 25, 'feature_fraction': 0.7814141590349598, 'bagging_fraction': 0.8267618613151968, 'min_child_samples': 41}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=2, value=None),
#  FrozenTrial(number=49, state=1, values=[0.029054402458894454], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 53, 702888), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 59, 175981), params={'n_estimators': 389, 'learning_rate': 0.0668955562533266, 'num_leaves': 59, 'feature_fraction': 0.6814043348769654, 'bagging_fraction': 0.6763133759295992, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=49, value=None),
#  FrozenTrial(number=11, state=1, values=[0.02914104508221635], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 14, 334885), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 15, 530709), params={'n_estimators': 211, 'learning_rate': 0.03141149793270349, 'num_leaves': 20, 'feature_fraction': 0.9987486962553719, 'bagging_fraction': 0.9629588713285726, 'min_child_samples': 36}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=11, value=None),
#  FrozenTrial(number=33, state=1, values=[0.029178045344780947], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 25, 204269), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 32, 458310), params={'n_estimators': 331, 'learning_rate': 0.04752821673074341, 'num_leaves': 117, 'feature_fraction': 0.9716287551107361, 'bagging_fraction': 0.9426989234553907, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=33, value=None),
#  FrozenTrial(number=3, state=1, values=[0.029221790641272845], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 49, 753803), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 51, 561455), params={'n_estimators': 325, 'learning_rate': 0.049320777344478615, 'num_leaves': 99, 'feature_fraction': 0.669213984986861, 'bagging_fraction': 0.7732879929978973, 'min_child_samples': 20}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=3, value=None),
#  FrozenTrial(number=4, state=1, values=[0.029265673130812693], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 51, 562641), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 53, 146467), params={'n_estimators': 380, 'learning_rate': 0.04528550744289921, 'num_leaves': 95, 'feature_fraction': 0.6797829378305154, 'bagging_fraction': 0.7871718745221721, 'min_child_samples': 28}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=4, value=None),
#  FrozenTrial(number=5, state=1, values=[0.02927501154785497], datetime_start=datetime.datetime(2025, 8, 5, 14, 6, 53, 147724), datetime_complete=datetime.datetime(2025, 8, 5, 14, 6, 53, 977115), params={'n_estimators': 399, 'learning_rate': 0.08329632458521416, 'num_leaves': 91, 'feature_fraction': 0.4977409009598658, 'bagging_fraction': 0.7603550581908067, 'min_child_samples': 48}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=5, value=None),
#  FrozenTrial(number=17, state=1, values=[0.029327560328445825], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 23, 470276), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 24, 69877), params={'n_estimators': 236, 'learning_rate': 0.057674462637247036, 'num_leaves': 53, 'feature_fraction': 0.7217604899914896, 'bagging_fraction': 0.5538853318390747, 'min_child_samples': 43}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=17, value=None),
#  FrozenTrial(number=30, state=1, values=[0.0293539954237006], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 16, 767895), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 18, 407566), params={'n_estimators': 353, 'learning_rate': 0.05348196609147154, 'num_leaves': 103, 'feature_fraction': 0.4972894335565135, 'bagging_fraction': 0.6410293082534345, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=30, value=None),
#  FrozenTrial(number=41, state=1, values=[0.029385856599881986], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 6, 430093), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 13, 806249), params={'n_estimators': 476, 'learning_rate': 0.06402244345378338, 'num_leaves': 90, 'feature_fraction': 0.6991707921684156, 'bagging_fraction': 0.8094870991229705, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=41, value=None),
#  FrozenTrial(number=27, state=1, values=[0.029449119584822728], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 0, 824665), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 3, 917311), params={'n_estimators': 369, 'learning_rate': 0.07414030223743999, 'num_leaves': 84, 'feature_fraction': 0.7302079413036106, 'bagging_fraction': 0.5581516164375521, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=27, value=None),
#  FrozenTrial(number=24, state=1, values=[0.02946435913635522], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 48, 283672), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 54, 933621), params={'n_estimators': 415, 'learning_rate': 0.060788302535630806, 'num_leaves': 109, 'feature_fraction': 0.6297216338317, 'bagging_fraction': 0.6916067714677392, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=24, value=None),
#  FrozenTrial(number=42, state=1, values=[0.029494331312043074], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 13, 807390), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 22, 559063), params={'n_estimators': 429, 'learning_rate': 0.0838612552363649, 'num_leaves': 98, 'feature_fraction': 0.7668753631418647, 'bagging_fraction': 0.9294534241496355, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=42, value=None),
#  FrozenTrial(number=47, state=1, values=[0.02973966824264036], datetime_start=datetime.datetime(2025, 8, 5, 14, 9, 46, 91007), datetime_complete=datetime.datetime(2025, 8, 5, 14, 9, 50, 142091), params={'n_estimators': 418, 'learning_rate': 0.08784702672781902, 'num_leaves': 69, 'feature_fraction': 0.7175405215293226, 'bagging_fraction': 0.9128645920443824, 'min_child_samples': 15}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=47, value=None),
#  FrozenTrial(number=28, state=1, values=[0.02983628465532831], datetime_start=datetime.datetime(2025, 8, 5, 14, 8, 3, 918481), datetime_complete=datetime.datetime(2025, 8, 5, 14, 8, 12, 65593), params={'n_estimators': 394, 'learning_rate': 0.09308871923007676, 'num_leaves': 69, 'feature_fraction': 0.5705435049200277, 'bagging_fraction': 0.7080042580562933, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=28, value=None),
#  FrozenTrial(number=15, state=1, values=[0.03003040581350591], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 18, 932739), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 21, 956915), params={'n_estimators': 431, 'learning_rate': 0.09790607727425152, 'num_leaves': 60, 'feature_fraction': 0.7480285870919989, 'bagging_fraction': 0.4877430597837541, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=15, value=None),
#  FrozenTrial(number=10, state=1, values=[0.031325253376945826], datetime_start=datetime.datetime(2025, 8, 5, 14, 7, 12, 970766), datetime_complete=datetime.datetime(2025, 8, 5, 14, 7, 14, 333872), params={'n_estimators': 265, 'learning_rate': 0.010115014849724081, 'num_leaves': 48, 'feature_fraction': 0.8049220202680232, 'bagging_fraction': 0.5165742176001858, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=10, value=None)]

In [ ]:
# desc_tc = desc_test[common_cols4]
# top_trials = sorted_tc_light[:15]
# all_predictions = []
# TC_light = []
# for i, trial in enumerate(top_trials):
#     params = trial.params
#     params['objective'] = 'mae'
#     params['metric'] = 'mae'
#     params['bagging_freq'] = 1
#     params['n_jobs'] = -1
#     params['verbosity'] = -1
#     params['random_state'] = 42
#     model = Model_light(params)
#     model.train(x_tc, y_tc)
#     preds = model.evalution()
#     all_predictions.append(preds)
#     TC_light.append(model.prediction(desc_tc))
#     print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
# final_ensemble_prediction = np.mean(all_predictions, axis=0)
# final_TC_light = np.mean(TC_light, axis=0)
# print(final_ensemble_prediction)
# print(final_TC_light)

**Density**

In [ ]:
train_cols5 = set(density.columns) - {"Density"}
test_cols5 = set(desc_test.columns)
common_cols5 = list(train_cols5& test_cols5)
x_density=density[common_cols5].copy()
y_density=density["Density"].copy()

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective(trial, x_density, y_density), n_trials = 50)

In [ ]:
# completed_trials_de = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_de = sorted(completed_trials_de, key =lambda t: t.value)

In [ ]:
sorted_de = [FrozenTrial(number=36, state=1, values=[0.06545450841557454], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 40, 232451), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 42, 489262), params={'n_estimators': 330, 'learning_rate': 0.08431197110565065, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=36, value=None),
 FrozenTrial(number=31, state=1, values=[0.06566270021423308], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 31, 471836), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 33, 391303), params={'n_estimators': 267, 'learning_rate': 0.06801761051899402, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=31, value=None),
 FrozenTrial(number=42, state=1, values=[0.06590316926046713], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 53, 619872), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 55, 508462), params={'n_estimators': 294, 'learning_rate': 0.07988497687901512, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=42, value=None),
 FrozenTrial(number=30, state=1, values=[0.06596902007009586], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 29, 512103), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 31, 469817), params={'n_estimators': 265, 'learning_rate': 0.06377034464009078, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=30, value=None),
 FrozenTrial(number=43, state=1, values=[0.0662142406213371], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 55, 510743), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 58, 571632), params={'n_estimators': 496, 'learning_rate': 0.07908699509776845, 'max_depth': 4, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=43, value=None),
 FrozenTrial(number=41, state=1, values=[0.06640145806005274], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 50, 127566), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 53, 616568), params={'n_estimators': 266, 'learning_rate': 0.06695934175610303, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=41, value=None),
 FrozenTrial(number=34, state=1, values=[0.06646751314696625], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 36, 621005), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 38, 644340), params={'n_estimators': 297, 'learning_rate': 0.08478667987986115, 'max_depth': 3, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=34, value=None),
 FrozenTrial(number=32, state=1, values=[0.06649757147662502], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 33, 392457), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 35, 274186), params={'n_estimators': 262, 'learning_rate': 0.06785500626659698, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=32, value=None),
 FrozenTrial(number=45, state=1, values=[0.06670767883642728], datetime_start=datetime.datetime(2025, 8, 5, 14, 13, 1, 465700), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 4, 602271), params={'n_estimators': 458, 'learning_rate': 0.07898920444694216, 'max_depth': 4, 'min_child_weight': 9, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=45, value=None),
 FrozenTrial(number=44, state=1, values=[0.06677611070944593], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 58, 572901), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 1, 464474), params={'n_estimators': 491, 'learning_rate': 0.07755866983040852, 'max_depth': 4, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=44, value=None),
 FrozenTrial(number=47, state=1, values=[0.06687263144166873], datetime_start=datetime.datetime(2025, 8, 5, 14, 13, 6, 228314), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 8, 751808), params={'n_estimators': 412, 'learning_rate': 0.09386870242661605, 'max_depth': 4, 'min_child_weight': 10, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=47, value=None),
 FrozenTrial(number=38, state=1, values=[0.06710645819311847], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 44, 85935), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 47, 44249), params={'n_estimators': 369, 'learning_rate': 0.08765313081337145, 'max_depth': 3, 'min_child_weight': 6, 'gamma': 0, 'reg_lambda': 6, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=38, value=None),
 FrozenTrial(number=2, state=1, values=[0.06804391550989441], datetime_start=datetime.datetime(2025, 8, 5, 14, 10, 57, 949110), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 2, 611862), params={'n_estimators': 274, 'learning_rate': 0.03556668677058889, 'max_depth': 6, 'min_child_weight': 4, 'gamma': 0, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=2, value=None),
 FrozenTrial(number=22, state=1, values=[0.0683931552683862], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 1, 925056), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 8, 111544), params={'n_estimators': 232, 'learning_rate': 0.035654066894296614, 'max_depth': 7, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=22, value=None),
 FrozenTrial(number=26, state=1, values=[0.06904823632466726], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 20, 550561), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 24, 570254), params={'n_estimators': 284, 'learning_rate': 0.036238963567608466, 'max_depth': 6, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=26, value=None),
 FrozenTrial(number=17, state=1, values=[0.06968157717556928], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 40, 404462), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 48, 773874), params={'n_estimators': 244, 'learning_rate': 0.023123165758828404, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=17, value=None),
 FrozenTrial(number=24, state=1, values=[0.06979334945257995], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 9, 417092), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 16, 818255), params={'n_estimators': 230, 'learning_rate': 0.023421577705615863, 'max_depth': 9, 'min_child_weight': 8, 'gamma': 0, 'reg_lambda': 4, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=24, value=None),
 FrozenTrial(number=19, state=1, values=[0.0702636256681339], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 50, 366253), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 55, 487784), params={'n_estimators': 239, 'learning_rate': 0.05306179265929079, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=19, value=None),
 FrozenTrial(number=21, state=1, values=[0.0706575281865111], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 57, 65712), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 1, 922645), params={'n_estimators': 241, 'learning_rate': 0.054613173554345716, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 0, 'reg_lambda': 2, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=21, value=None),
 FrozenTrial(number=14, state=1, values=[0.0727574183080608], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 32, 661019), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 36, 789172), params={'n_estimators': 251, 'learning_rate': 0.021598997562929996, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 0, 'reg_lambda': 3, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=14, value=None),
 FrozenTrial(number=12, state=1, values=[0.07539680497438034], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 26, 863559), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 30, 978561), params={'n_estimators': 218, 'learning_rate': 0.016468214053917022, 'max_depth': 5, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=12, value=None),
 FrozenTrial(number=10, state=1, values=[0.07573366775355649], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 14, 501798), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 21, 684310), params={'n_estimators': 224, 'learning_rate': 0.011779445658343744, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=10, value=None),
 FrozenTrial(number=11, state=1, values=[0.07719743353137483], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 21, 687034), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 26, 860745), params={'n_estimators': 203, 'learning_rate': 0.011583053579516348, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 0, 'reg_lambda': 5, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=11, value=None),
 FrozenTrial(number=37, state=1, values=[0.07803206451309244], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 42, 490660), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 44, 84726), params={'n_estimators': 334, 'learning_rate': 0.08491533232780774, 'max_depth': 4, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=37, value=None),
 FrozenTrial(number=20, state=1, values=[0.07842811328529446], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 55, 489497), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 57, 62971), params={'n_estimators': 284, 'learning_rate': 0.06050467385519081, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=20, value=None),
 FrozenTrial(number=46, state=1, values=[0.07924014882372035], datetime_start=datetime.datetime(2025, 8, 5, 14, 13, 4, 603592), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 6, 225879), params={'n_estimators': 309, 'learning_rate': 0.07249263090320704, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=46, value=None),
 FrozenTrial(number=25, state=1, values=[0.08004912219612526], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 16, 820619), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 20, 549389), params={'n_estimators': 340, 'learning_rate': 0.023556193856657717, 'max_depth': 7, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 2, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=25, value=None),
 FrozenTrial(number=3, state=1, values=[0.08110821254041113], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 2, 613243), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 4, 118037), params={'n_estimators': 312, 'learning_rate': 0.09789633597955605, 'max_depth': 7, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=3, value=None),
 FrozenTrial(number=39, state=1, values=[0.0811104071842615], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 47, 45893), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 48, 579554), params={'n_estimators': 316, 'learning_rate': 0.08167142370377724, 'max_depth': 4, 'min_child_weight': 10, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=39, value=None),
 FrozenTrial(number=49, state=1, values=[0.08144980150765493], datetime_start=datetime.datetime(2025, 8, 5, 14, 13, 10, 298533), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 11, 930375), params={'n_estimators': 344, 'learning_rate': 0.0715826710373941, 'max_depth': 5, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 4, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=49, value=None),
 FrozenTrial(number=33, state=1, values=[0.08166015735059962], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 35, 275624), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 36, 619644), params={'n_estimators': 264, 'learning_rate': 0.06819110501831246, 'max_depth': 3, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=33, value=None),
 FrozenTrial(number=35, state=1, values=[0.08305983406014478], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 38, 645276), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 40, 231057), params={'n_estimators': 299, 'learning_rate': 0.08802457561317505, 'max_depth': 3, 'min_child_weight': 6, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=35, value=None),
 FrozenTrial(number=23, state=1, values=[0.0835001321577328], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 8, 114855), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 9, 414566), params={'n_estimators': 200, 'learning_rate': 0.03584318488322902, 'max_depth': 7, 'min_child_weight': 9, 'gamma': 1, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=23, value=None),
 FrozenTrial(number=5, state=1, values=[0.08787755661498961], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 6, 294474), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 7, 950144), params={'n_estimators': 327, 'learning_rate': 0.07601570970111139, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 1, 'reg_lambda': 9, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=5, value=None),
 FrozenTrial(number=16, state=1, values=[0.08811116608126728], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 38, 754070), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 40, 403109), params={'n_estimators': 301, 'learning_rate': 0.06339195702461296, 'max_depth': 5, 'min_child_weight': 3, 'gamma': 1, 'reg_lambda': 3, 'reg_alpha': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=16, value=None),
 FrozenTrial(number=29, state=1, values=[0.08876318633787039], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 27, 591576), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 29, 509720), params={'n_estimators': 378, 'learning_rate': 0.03268917688260331, 'max_depth': 6, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=29, value=None),
 FrozenTrial(number=13, state=1, values=[0.08957718915079335], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 30, 981639), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 32, 658927), params={'n_estimators': 275, 'learning_rate': 0.029735391449047396, 'max_depth': 5, 'min_child_weight': 2, 'gamma': 1, 'reg_lambda': 7, 'reg_alpha': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=13, value=None),
 FrozenTrial(number=18, state=1, values=[0.09186889503825797], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 48, 776836), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 50, 364866), params={'n_estimators': 300, 'learning_rate': 0.04100262534360381, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 2, 'reg_lambda': 1, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=18, value=None),
 FrozenTrial(number=15, state=1, values=[0.09239968750682803], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 36, 790588), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 38, 752502), params={'n_estimators': 399, 'learning_rate': 0.03980284474598374, 'max_depth': 7, 'min_child_weight': 3, 'gamma': 2, 'reg_lambda': 3, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=15, value=None),
 FrozenTrial(number=27, state=1, values=[0.0938270705431068], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 24, 573045), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 26, 18126), params={'n_estimators': 281, 'learning_rate': 0.03779087672052168, 'max_depth': 6, 'min_child_weight': 10, 'gamma': 2, 'reg_lambda': 4, 'reg_alpha': 6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=27, value=None),
 FrozenTrial(number=0, state=1, values=[0.09566918653952192], datetime_start=datetime.datetime(2025, 8, 5, 14, 10, 54, 707036), datetime_complete=datetime.datetime(2025, 8, 5, 14, 10, 56, 549377), params={'n_estimators': 370, 'learning_rate': 0.0309667481490603, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 2, 'reg_lambda': 2, 'reg_alpha': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=9, state=1, values=[0.09733467076277769], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 12, 932498), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 14, 500397), params={'n_estimators': 356, 'learning_rate': 0.07485967928019557, 'max_depth': 4, 'min_child_weight': 10, 'gamma': 3, 'reg_lambda': 9, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=9, value=None),
 FrozenTrial(number=48, state=1, values=[0.0979803400205656], datetime_start=datetime.datetime(2025, 8, 5, 14, 13, 8, 754521), datetime_complete=datetime.datetime(2025, 8, 5, 14, 13, 10, 296058), params={'n_estimators': 324, 'learning_rate': 0.06077647379376519, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 3, 'reg_lambda': 3, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=48, value=None),
 FrozenTrial(number=4, state=1, values=[0.09800227653231194], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 4, 119484), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 6, 293120), params={'n_estimators': 473, 'learning_rate': 0.047487124594103196, 'max_depth': 10, 'min_child_weight': 5, 'gamma': 3, 'reg_lambda': 6, 'reg_alpha': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=4, value=None),
 FrozenTrial(number=1, state=1, values=[0.10061349810307359], datetime_start=datetime.datetime(2025, 8, 5, 14, 10, 56, 552366), datetime_complete=datetime.datetime(2025, 8, 5, 14, 10, 57, 947800), params={'n_estimators': 270, 'learning_rate': 0.046502367577527765, 'max_depth': 10, 'min_child_weight': 7, 'gamma': 4, 'reg_lambda': 1, 'reg_alpha': 0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=1, value=None),
 FrozenTrial(number=6, state=1, values=[0.10222149401308508], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 7, 951590), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 9, 992951), params={'n_estimators': 467, 'learning_rate': 0.028229574677440777, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 3, 'reg_lambda': 3, 'reg_alpha': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=6, value=None),
 FrozenTrial(number=40, state=1, values=[0.1031039563506487], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 48, 581103), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 50, 125971), params={'n_estimators': 357, 'learning_rate': 0.09194280447464052, 'max_depth': 3, 'min_child_weight': 5, 'gamma': 4, 'reg_lambda': 6, 'reg_alpha': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=40, value=None),
 FrozenTrial(number=28, state=1, values=[0.10549182126893751], datetime_start=datetime.datetime(2025, 8, 5, 14, 12, 26, 20476), datetime_complete=datetime.datetime(2025, 8, 5, 14, 12, 27, 589927), params={'n_estimators': 315, 'learning_rate': 0.04627904598405796, 'max_depth': 4, 'min_child_weight': 9, 'gamma': 4, 'reg_lambda': 7, 'reg_alpha': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=28, value=None),
 FrozenTrial(number=8, state=1, values=[0.1059160288970196], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 11, 148730), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 12, 931172), params={'n_estimators': 438, 'learning_rate': 0.07418721300325454, 'max_depth': 8, 'min_child_weight': 8, 'gamma': 5, 'reg_lambda': 9, 'reg_alpha': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=8, value=None),
 FrozenTrial(number=7, state=1, values=[0.10647351509265802], datetime_start=datetime.datetime(2025, 8, 5, 14, 11, 9, 994338), datetime_complete=datetime.datetime(2025, 8, 5, 14, 11, 11, 147390), params={'n_estimators': 257, 'learning_rate': 0.09798778841627573, 'max_depth': 6, 'min_child_weight': 5, 'gamma': 5, 'reg_lambda': 10, 'reg_alpha': 3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'gamma': IntDistribution(high=5, log=False, low=0, step=1), 'reg_lambda': IntDistribution(high=10, log=False, low=1, step=1), 'reg_alpha': IntDistribution(high=10, log=False, low=0, step=1)}, trial_id=7, value=None)]

In [ ]:
desc_de = desc_test[common_cols5]
top_trials = sorted_de[:10]
all_predictions = []
DE = []
for i, trial in enumerate(top_trials):
    params = trial.params
    params['random_state'] = 42+i
    model = Model_XG(params)
    model.train(x_density, y_density)
    preds = model.evalution()
    all_predictions.append(preds)
    DE.append(model.prediction(desc_de))
    print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
final_ensemble_prediction = np.mean(all_predictions, axis=0)
final_DE = np.mean(DE, axis=0)
print(final_ensemble_prediction)
print(final_DE)

In [ ]:
# study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler())
# study.optimize(lambda trial: objective_light(trial, x_density, y_density), n_trials = 50)

In [ ]:
# completed_trials_de_light = [t for t in study.trials if t.state == TrialState.COMPLETE]
# sorted_trials_de_light = sorted(completed_trials_de_light, key =lambda t: t.value)

In [ ]:
# sorted_de_light = [FrozenTrial(number=48, state=1, values=[0.07225440090414846], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 20, 984278), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 21, 922313), params={'n_estimators': 212, 'learning_rate': 0.02033407692407902, 'num_leaves': 20, 'feature_fraction': 0.6398864766736689, 'bagging_fraction': 0.46096049320492066, 'min_child_samples': 33}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=48, value=None),
#  FrozenTrial(number=22, state=1, values=[0.07239412849730832], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 23, 844982), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 25, 375236), params={'n_estimators': 336, 'learning_rate': 0.010051792873958276, 'num_leaves': 81, 'feature_fraction': 0.5816597830880129, 'bagging_fraction': 0.49411315671646816, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=22, value=None),
#  FrozenTrial(number=43, state=1, values=[0.07325263638840217], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 11, 962385), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 13, 72235), params={'n_estimators': 204, 'learning_rate': 0.020593701678241698, 'num_leaves': 20, 'feature_fraction': 0.5409951652656653, 'bagging_fraction': 0.45017935366809275, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=43, value=None),
#  FrozenTrial(number=11, state=1, values=[0.07397951027040617], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 58, 293436), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 59, 672735), params={'n_estimators': 205, 'learning_rate': 0.011377195321561537, 'num_leaves': 78, 'feature_fraction': 0.6145136814830062, 'bagging_fraction': 0.5402737207611533, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=11, value=None),
#  FrozenTrial(number=14, state=1, values=[0.07401730792125064], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 3, 900366), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 5, 736398), params={'n_estimators': 294, 'learning_rate': 0.010704470511853557, 'num_leaves': 64, 'feature_fraction': 0.56904270837979, 'bagging_fraction': 0.517226832288341, 'min_child_samples': 21}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=14, value=None),
#  FrozenTrial(number=45, state=1, values=[0.07448583326226513], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 14, 939345), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 16, 95246), params={'n_estimators': 209, 'learning_rate': 0.021356957591187824, 'num_leaves': 45, 'feature_fraction': 0.6028225891396471, 'bagging_fraction': 0.44097836943081065, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=45, value=None),
#  FrozenTrial(number=16, state=1, values=[0.07458906473280154], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 7, 546709), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 9, 401911), params={'n_estimators': 297, 'learning_rate': 0.011453586077755274, 'num_leaves': 92, 'feature_fraction': 0.5924305800002994, 'bagging_fraction': 0.5415860932561571, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=16, value=None),
#  FrozenTrial(number=8, state=1, values=[0.07463833116250276], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 54, 99090), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 55, 768813), params={'n_estimators': 236, 'learning_rate': 0.017349521325766122, 'num_leaves': 25, 'feature_fraction': 0.5123402355129648, 'bagging_fraction': 0.4743808710780551, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=8, value=None),
#  FrozenTrial(number=35, state=1, values=[0.07479866249916113], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 54, 45383), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 55, 193994), params={'n_estimators': 256, 'learning_rate': 0.022830236346159084, 'num_leaves': 73, 'feature_fraction': 0.505670190906796, 'bagging_fraction': 0.5462421270566954, 'min_child_samples': 32}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=35, value=None),
#  FrozenTrial(number=21, state=1, values=[0.074938586066156], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 21, 768314), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 23, 843554), params={'n_estimators': 277, 'learning_rate': 0.010851725394910637, 'num_leaves': 88, 'feature_fraction': 0.6208173212333997, 'bagging_fraction': 0.570651665426464, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=21, value=None),
#  FrozenTrial(number=27, state=1, values=[0.07516924807110006], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 31, 596651), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 34, 285280), params={'n_estimators': 453, 'learning_rate': 0.010021323658793492, 'num_leaves': 100, 'feature_fraction': 0.6186872624991379, 'bagging_fraction': 0.42372167808649724, 'min_child_samples': 18}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=27, value=None),
#  FrozenTrial(number=41, state=1, values=[0.07532806123549755], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 8, 331415), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 10, 318493), params={'n_estimators': 247, 'learning_rate': 0.017999871736500472, 'num_leaves': 27, 'feature_fraction': 0.4984183738385208, 'bagging_fraction': 0.4884283227566852, 'min_child_samples': 8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=41, value=None),
#  FrozenTrial(number=9, state=1, values=[0.0753947541538956], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 55, 770087), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 57, 530537), params={'n_estimators': 245, 'learning_rate': 0.011182067453535597, 'num_leaves': 76, 'feature_fraction': 0.4092319383435576, 'bagging_fraction': 0.6270826738579447, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=9, value=None),
#  FrozenTrial(number=44, state=1, values=[0.07571251946857975], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 13, 74442), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 14, 937173), params={'n_estimators': 202, 'learning_rate': 0.014428860160566593, 'num_leaves': 95, 'feature_fraction': 0.5522167833249492, 'bagging_fraction': 0.7324988259598743, 'min_child_samples': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=44, value=None),
#  FrozenTrial(number=10, state=1, values=[0.07579854577902545], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 57, 531885), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 58, 292354), params={'n_estimators': 216, 'learning_rate': 0.04130447517403512, 'num_leaves': 26, 'feature_fraction': 0.652951829245333, 'bagging_fraction': 0.4014968005688808, 'min_child_samples': 38}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=10, value=None),
#  FrozenTrial(number=42, state=1, values=[0.07588684553635065], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 10, 320598), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 11, 960223), params={'n_estimators': 236, 'learning_rate': 0.014935186985673147, 'num_leaves': 37, 'feature_fraction': 0.44560531380071217, 'bagging_fraction': 0.5473939081402454, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=42, value=None),
#  FrozenTrial(number=31, state=1, values=[0.07613087997010207], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 40, 159940), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 42, 117951), params={'n_estimators': 296, 'learning_rate': 0.015320550186109689, 'num_leaves': 91, 'feature_fraction': 0.581848973818102, 'bagging_fraction': 0.5685226821828512, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=31, value=None),
#  FrozenTrial(number=23, state=1, values=[0.07648382509629403], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 25, 376855), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 26, 818779), params={'n_estimators': 336, 'learning_rate': 0.025308259038379854, 'num_leaves': 81, 'feature_fraction': 0.5773280033434846, 'bagging_fraction': 0.46759838876226867, 'min_child_samples': 30}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=23, value=None),
#  FrozenTrial(number=46, state=1, values=[0.07660904664303009], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 16, 96902), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 19, 960832), params={'n_estimators': 217, 'learning_rate': 0.021554294948619834, 'num_leaves': 44, 'feature_fraction': 0.9651614504240906, 'bagging_fraction': 0.4422630629732169, 'min_child_samples': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=46, value=None),
#  FrozenTrial(number=24, state=1, values=[0.07681084566926027], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 26, 820019), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 29, 139515), params={'n_estimators': 392, 'learning_rate': 0.015931591366142986, 'num_leaves': 66, 'feature_fraction': 0.6722067083313973, 'bagging_fraction': 0.5857160714745615, 'min_child_samples': 31}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=24, value=None),
#  FrozenTrial(number=28, state=1, values=[0.07691147461476823], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 34, 286439), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 35, 521193), params={'n_estimators': 313, 'learning_rate': 0.024505250185800763, 'num_leaves': 57, 'feature_fraction': 0.4700135071627496, 'bagging_fraction': 0.6250295953503314, 'min_child_samples': 40}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=28, value=None),
#  FrozenTrial(number=37, state=1, values=[0.0769661394164925], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 58, 558390), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 0, 429544), params={'n_estimators': 230, 'learning_rate': 0.02115328179280597, 'num_leaves': 104, 'feature_fraction': 0.5870002528193888, 'bagging_fraction': 0.44288980474334777, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=37, value=None),
#  FrozenTrial(number=30, state=1, values=[0.0770022860335808], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 38, 308170), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 40, 158797), params={'n_estimators': 373, 'learning_rate': 0.017644877451520824, 'num_leaves': 47, 'feature_fraction': 0.48423023211869287, 'bagging_fraction': 0.5153685624850419, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=30, value=None),
#  FrozenTrial(number=25, state=1, values=[0.07712335031693199], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 29, 140739), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 30, 262097), params={'n_estimators': 223, 'learning_rate': 0.030641342318936005, 'num_leaves': 130, 'feature_fraction': 0.5170278477287237, 'bagging_fraction': 0.47380525062687534, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=25, value=None),
#  FrozenTrial(number=7, state=1, values=[0.07734857344181677], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 52, 733638), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 54, 97572), params={'n_estimators': 375, 'learning_rate': 0.022509226827621263, 'num_leaves': 40, 'feature_fraction': 0.4397347627871709, 'bagging_fraction': 0.7333042324288019, 'min_child_samples': 49}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=7, value=None),
#  FrozenTrial(number=47, state=1, values=[0.0781417740391693], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 19, 961991), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 20, 983428), params={'n_estimators': 206, 'learning_rate': 0.044552622101938874, 'num_leaves': 29, 'feature_fraction': 0.6567936614635527, 'bagging_fraction': 0.42168084869193434, 'min_child_samples': 27}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=47, value=None),
#  FrozenTrial(number=26, state=1, values=[0.07876151466669677], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 30, 263391), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 31, 595412), params={'n_estimators': 263, 'learning_rate': 0.039706279333313825, 'num_leaves': 78, 'feature_fraction': 0.7381988653476983, 'bagging_fraction': 0.5220631352797711, 'min_child_samples': 34}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=26, value=None),
#  FrozenTrial(number=12, state=1, values=[0.079180627814523], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 59, 674052), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 0, 853752), params={'n_estimators': 203, 'learning_rate': 0.039547093953054724, 'num_leaves': 83, 'feature_fraction': 0.6255392277907034, 'bagging_fraction': 0.5327615311280157, 'min_child_samples': 28}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=12, value=None),
#  FrozenTrial(number=39, state=1, values=[0.07926918008648122], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 2, 714760), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 4, 804999), params={'n_estimators': 405, 'learning_rate': 0.028120534431797278, 'num_leaves': 116, 'feature_fraction': 0.831389128538913, 'bagging_fraction': 0.4031905148920692, 'min_child_samples': 27}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=39, value=None),
#  FrozenTrial(number=32, state=1, values=[0.07940592753270913], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 42, 119075), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 49, 151422), params={'n_estimators': 275, 'learning_rate': 0.010750021520187666, 'num_leaves': 86, 'feature_fraction': 0.5998040385012401, 'bagging_fraction': 0.5067115981321916, 'min_child_samples': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=32, value=None),
#  FrozenTrial(number=36, state=1, values=[0.07958025011246642], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 55, 195178), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 58, 557189), params={'n_estimators': 431, 'learning_rate': 0.0146544831148792, 'num_leaves': 68, 'feature_fraction': 0.7062784513052679, 'bagging_fraction': 0.6737555357879373, 'min_child_samples': 27}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=36, value=None),
#  FrozenTrial(number=18, state=1, values=[0.08127178999304421], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 11, 257538), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 13, 153338), params={'n_estimators': 320, 'learning_rate': 0.03418153983914652, 'num_leaves': 98, 'feature_fraction': 0.5659631893561331, 'bagging_fraction': 0.779001809499799, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=18, value=None),
#  FrozenTrial(number=49, state=1, values=[0.08142449960853244], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 21, 923819), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 22, 962810), params={'n_estimators': 261, 'learning_rate': 0.06633210452367967, 'num_leaves': 20, 'feature_fraction': 0.6496216450807357, 'bagging_fraction': 0.4621719922323462, 'min_child_samples': 38}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=49, value=None),
#  FrozenTrial(number=34, state=1, values=[0.08211303602967975], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 51, 795872), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 54, 44266), params={'n_estimators': 299, 'learning_rate': 0.028329460770545312, 'num_leaves': 113, 'feature_fraction': 0.5434500205965667, 'bagging_fraction': 0.7038263257103828, 'min_child_samples': 24}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=34, value=None),
#  FrozenTrial(number=1, state=1, values=[0.08241914658684296], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 20, 696446), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 23, 376325), params={'n_estimators': 254, 'learning_rate': 0.027244476114229442, 'num_leaves': 49, 'feature_fraction': 0.40708602527680116, 'bagging_fraction': 0.7102489696948087, 'min_child_samples': 11}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=1, value=None),
#  FrozenTrial(number=3, state=1, values=[0.08308703718416438], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 29, 389722), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 32, 465080), params={'n_estimators': 300, 'learning_rate': 0.02035836005262298, 'num_leaves': 122, 'feature_fraction': 0.4576211784111679, 'bagging_fraction': 0.6752957866317362, 'min_child_samples': 13}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=3, value=None),
#  FrozenTrial(number=4, state=1, values=[0.08319210021478886], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 32, 465973), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 35, 823830), params={'n_estimators': 349, 'learning_rate': 0.05131225762068709, 'num_leaves': 36, 'feature_fraction': 0.5369683616259673, 'bagging_fraction': 0.8379720629767355, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=4, value=None),
#  FrozenTrial(number=15, state=1, values=[0.08333471201510427], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 5, 737861), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 7, 545485), params={'n_estimators': 295, 'learning_rate': 0.05872869438217819, 'num_leaves': 64, 'feature_fraction': 0.7549364979686041, 'bagging_fraction': 0.5999355568102733, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=15, value=None),
#  FrozenTrial(number=38, state=1, values=[0.08359983514293537], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 0, 430558), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 2, 713289), params={'n_estimators': 362, 'learning_rate': 0.03554328991519447, 'num_leaves': 54, 'feature_fraction': 0.6276248192457546, 'bagging_fraction': 0.49845303076013664, 'min_child_samples': 22}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=38, value=None),
#  FrozenTrial(number=13, state=1, values=[0.08377088618115544], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 0, 855373), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 3, 899443), params={'n_estimators': 496, 'learning_rate': 0.03198918658823342, 'num_leaves': 90, 'feature_fraction': 0.7633172821767586, 'bagging_fraction': 0.5310322391206483, 'min_child_samples': 29}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=13, value=None),
#  FrozenTrial(number=20, state=1, values=[0.08519783752548986], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 20, 31435), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 21, 767083), params={'n_estimators': 322, 'learning_rate': 0.09896369013306333, 'num_leaves': 72, 'feature_fraction': 0.8150487481470032, 'bagging_fraction': 0.6469514934410676, 'min_child_samples': 43}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=20, value=None),
#  FrozenTrial(number=19, state=1, values=[0.08549529844072148], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 13, 154732), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 20, 30161), params={'n_estimators': 474, 'learning_rate': 0.04903479159522793, 'num_leaves': 56, 'feature_fraction': 0.6930914449135384, 'bagging_fraction': 0.9513324143353884, 'min_child_samples': 25}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=19, value=None),
#  FrozenTrial(number=6, state=1, values=[0.08682672528730342], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 40, 53244), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 52, 732185), params={'n_estimators': 361, 'learning_rate': 0.01902176528428673, 'num_leaves': 109, 'feature_fraction': 0.9972382397946623, 'bagging_fraction': 0.8626682471175262, 'min_child_samples': 10}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=6, value=None),
#  FrozenTrial(number=17, state=1, values=[0.08709833066102216], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 9, 403159), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 11, 256308), params={'n_estimators': 274, 'learning_rate': 0.07364761573705043, 'num_leaves': 70, 'feature_fraction': 0.701424927519306, 'bagging_fraction': 0.48042895297569665, 'min_child_samples': 21}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=17, value=None),
#  FrozenTrial(number=5, state=1, values=[0.08738246671147472], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 35, 825170), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 40, 52241), params={'n_estimators': 400, 'learning_rate': 0.05904989612435899, 'num_leaves': 61, 'feature_fraction': 0.4077074042563412, 'bagging_fraction': 0.8728091083914098, 'min_child_samples': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=5, value=None),
#  FrozenTrial(number=33, state=1, values=[0.08786945887940022], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 49, 152432), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 51, 794764), params={'n_estimators': 325, 'learning_rate': 0.08819590927364655, 'num_leaves': 95, 'feature_fraction': 0.6513857966212703, 'bagging_fraction': 0.5729687815457248, 'min_child_samples': 20}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=33, value=None),
#  FrozenTrial(number=2, state=1, values=[0.0879316416056586], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 23, 379070), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 29, 388231), params={'n_estimators': 419, 'learning_rate': 0.09485156478619142, 'num_leaves': 50, 'feature_fraction': 0.8934846211202175, 'bagging_fraction': 0.8945222233336223, 'min_child_samples': 23}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=2, value=None),
#  FrozenTrial(number=40, state=1, values=[0.08808913141300673], datetime_start=datetime.datetime(2025, 8, 5, 14, 31, 4, 806141), datetime_complete=datetime.datetime(2025, 8, 5, 14, 31, 8, 330411), params={'n_estimators': 284, 'learning_rate': 0.08417640196481144, 'num_leaves': 80, 'feature_fraction': 0.5289520995516368, 'bagging_fraction': 0.6008210135596536, 'min_child_samples': 12}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=40, value=None),
#  FrozenTrial(number=29, state=1, values=[0.08977608104583845], datetime_start=datetime.datetime(2025, 8, 5, 14, 30, 35, 522397), datetime_complete=datetime.datetime(2025, 8, 5, 14, 30, 38, 306911), params={'n_estimators': 345, 'learning_rate': 0.06897796992252102, 'num_leaves': 106, 'feature_fraction': 0.5512835538758944, 'bagging_fraction': 0.44799742473691384, 'min_child_samples': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=29, value=None),
#  FrozenTrial(number=0, state=1, values=[0.0908301647208722], datetime_start=datetime.datetime(2025, 8, 5, 14, 29, 14, 621274), datetime_complete=datetime.datetime(2025, 8, 5, 14, 29, 20, 695114), params={'n_estimators': 422, 'learning_rate': 0.078078055309181, 'num_leaves': 106, 'feature_fraction': 0.5058775509807881, 'bagging_fraction': 0.4263594485562731, 'min_child_samples': 7}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=200, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=130, log=False, low=20, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'min_child_samples': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None)]

In [ ]:
# desc_de = desc_test[common_cols5]
# top_trials = sorted_de_light[:15]
# all_predictions = []
# DE_light = []
# for i, trial in enumerate(top_trials):
#     params = trial.params
#     params['objective'] = 'mae'
#     params['metric'] = 'mae'
#     params['bagging_freq'] = 1
#     params['n_jobs'] = -1
#     params['verbosity'] = -1
#     params['random_state'] = 42
#     model = Model_light(params)
#     model.train(x_density, y_density)
#     preds = model.evalution()
#     all_predictions.append(preds)
#     DE_light.append(model.prediction(desc_de))
#     print(f"  Model {i+1}/{len(top_trials)} trained using params from Trial #{trial.number}.")
    
# final_ensemble_prediction = np.mean(all_predictions, axis=0)
# final_DE_light = np.mean(DE_light, axis=0)
# print(final_ensemble_prediction)
# print(final_DE_light)

# **Submission and Prediction**

In [ ]:
sub = {'id': Id,
    'Tg': final_TG,
    'FFV':     final_FFV,
    'Tc':      final_TC,
    'Density': final_DE,
    'Rg':      final_RG
}

In [ ]:
# sub_ = {'id': Id,
#     'Tg':      final_TG_light,
#     'FFV':     final_FFV_light,
#     'Tc':      final_TC_light,
#     'Density': final_DE_light,
#     'Rg':      final_RG_light
# }

In [ ]:
# submission1=pd.DataFrame(sub_)

In [ ]:
submission2=pd.DataFrame(sub)

In [ ]:
features = [col for col in submission2.columns if col != 'id']

In [ ]:
df_avg = submission2.copy()
# df_avg[features] = (submission2[features] + submission1[features])/2
df_avg[features] = submission2[features]

In [ ]:
df_avg.to_csv('submission.csv',index=False)